# 06 · What creates the pre-Haiyan nighttime-lights cycle?

**Question.** The 2012–7 November 2013 reliability-qualified NTL baseline rises and falls before Haiyan. Is the apparent cycle shared by bright and dim locations, related to observability or changing pixel composition, sensitive to interpolation or the assumed STL period, or consistent with a change in the underlying signal?

This notebook is a diagnostic extension of the previous Notebook 06. It retains the existing DNB-BRDF extraction, reference support, SC gate, map palette and principal baseline figures. It **does not project a business-as-usual baseline or estimate recovery**. A detected period or breakpoint is a finding to explain, not proof of lunar contamination, power demand or a pre-event outage.

**Read in order:** fixed support and observability → low-radiance days → independent dim/bright controls → periodicity without and with interpolation → STL sensitivity → optional BFAST → named Tacloban pixels. Run the source-data and cache cells once; plot and model cells can then be rerun separately.

The October 2026 research meetings motivated the dark-reference test and the distinction between a descriptive recurring component and a defensible physical explanation. The reference-period pixel selection is retrospective, so all earlier-2012 comparisons are descriptive rather than prospective validation.

## 1. Existing data route and fixed spatial supports

The characterization window begins on 19 January 2012 and ends before Haiyan. The 60-day reference selects positive pixels with at least three observed four-day composites; it does **not** define the whole characterization period. MQF = 0 supplies fresh/high-quality DNB-BRDF observations in this workflow. The regional daily P95 cap and the existing 10% SC gate remain unchanged. Ten percent is permissive: every plot must be read with its valid-pixel support.

The setup falls back to the original repository path when this deliverable is opened outside its `notebooks` directory. Place it beside the older notebook to use the original relative paths.

In [ ]:
from pathlib import Path
import json
import hashlib
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
from rasterio.enums import Resampling
from rasterio.features import rasterize
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (PROJECT_DIR / "datasets").exists(): PROJECT_DIR = Path('/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery')
DATA_DIR = PROJECT_DIR / "datasets"
PROCESSED_DIR = DATA_DIR / "VNP46/processed"
BASELINE_STORE = PROCESSED_DIR / "baseline/Haiyan_VNP46A2_baseline.zarr"
EXISTING_STORE = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"
GHSL_PATH = next(DATA_DIR.glob("**/GHSL_SMOD_E2015.tif"))
MUNICIPALITIES_PATH = DATA_DIR / "boundaries/MuniCities/MuniCities.shp"
OUTPUT_DIR = PROJECT_DIR / "output/baseline_characterization_v2"
FIGURE_DIR, TABLE_DIR = OUTPUT_DIR / "figures", OUTPUT_DIR / "tables"
for path in (FIGURE_DIR,TABLE_DIR): path.mkdir(parents=True,exist_ok=True)
EVENT_DATE = pd.Timestamp("2013-11-08")
BASELINE_DAYS, AGGREGATION_DAYS, KERNEL_SIZE = 60, 4, 5
BASELINE_START = EVENT_DATE-pd.Timedelta(days=BASELINE_DAYS)
PRE_EVENT_END = EVENT_DATE-pd.Timedelta(days=1)
CHARACTERIZATION_START = pd.Timestamp("2012-01-19")
CHARACTERIZATION_END = PRE_EVENT_END
GHSL_MASKS, SETTLEMENT_MASK = {"G3":(22,23,30)}, "G3"
SPATIAL_COMPLETENESS_PCT = 10.0
RQ_CLIP_PERCENTILE = 95.0
MIN_BASELINE_OBSERVATIONS = {1:5,4:3}
KNOWN_FILL_VALUES = (-9999.,-999.,-32768.,6553.5,65535.)
REBUILD_PROFILES = False  # True if source pixel values change without a date-axis change.
DNB_BAND, MQF_BAND = "DNB_BRDF_Corrected_NTL", "Mandatory_Quality_Flag"
GAP_FILLED_BAND = "Gap_Filled_DNB_BRDF_Corrected_NTL"

PLOT_FONT, PLOT_TEXT_COLOR, PLOT_GRID_COLOR = "Arial", "#243B5A", "#E8EDF3"
EVENT_COLOR, BASELINE_COLOR = "#0057FF", "#6C7882"
DNB_COLOR, GAP_FILLED_COLOR = "#0091FF", "#FF0000"
RQ_COLOR, RQ_FOUR_DAY_COLOR = "#00C54F", "#009227"
NTL_DISPLAY_MIN, NTL_DISPLAY_MAX = 0., 5.  # Display only; never truncate analytical values.
GHSL_CLASS_COLORS = {11:"#cdf57a",12:"#abcd66",13:"#375623",21:"#ffff00",22:"#a87000",23:"#732600",30:"#ff0000"}
GHSL_CLASS_LABELS = {11:"Very low density rural",12:"Low density rural",13:"Rural cluster",21:"Suburban / peri-urban",22:"Semi-dense urban",23:"Dense urban",30:"Urban centre"}
LOCATION_COLORS = {"Tacloban":"#0072B2","Palo":"#E69F00","Guiuan":"#009E73","Alangalang":"#CC79A7","Ormoc":"#D55E00","Baybay":"#56B4E9"}
SC_GREEN_COLORSCALE = [[0.,"rgba(255,255,255,0)"],[.2,"rgba(220,242,215,.25)"],
    [.4,"rgba(166,219,160,.45)"],[.6,"rgba(90,174,110,.65)"],[.8,"rgba(25,130,70,.82)"],[1.,"rgba(0,88,43,.98)"]]

def style_figure(fig,title,width=1280,height=720):
    fig.update_layout(template="plotly_white",paper_bgcolor="rgba(0,0,0,0)",plot_bgcolor="white",
        width=width,height=height,font=dict(family=PLOT_FONT,size=14,color=PLOT_TEXT_COLOR),
        title=dict(text=title,x=.03,xanchor="left",font=dict(size=22)),
        margin=dict(l=85,r=85,t=95,b=130),
        legend=dict(orientation="h",x=0,y=-.18,xanchor="left",yanchor="top"))
    fig.update_xaxes(zeroline=False,gridcolor=PLOT_GRID_COLOR)
    fig.update_yaxes(zeroline=False,gridcolor=PLOT_GRID_COLOR)
    return fig

def finish_figure(fig,filename):
    fig.write_html(FIGURE_DIR/f"{filename}.html",include_plotlyjs="directory")
    fig.show()

def add_haiyan_marker(fig,row=None,col=None):
    fig.add_vline(x=EVENT_DATE.to_pydatetime(),line_dash="dash",line_color=EVENT_COLOR,line_width=2,row=row,col=col)


In [ ]:
early = xr.open_zarr(BASELINE_STORE,chunks={},consolidated=False,mask_and_scale=False)
existing = xr.open_zarr(EXISTING_STORE,chunks={},consolidated=False,mask_and_scale=False)
if not np.all(early.processed.values == 1): raise RuntimeError("Finish Notebook 05 first.")
for axis in ("x","y"):
    np.testing.assert_allclose(early[axis].values,existing[axis].values,rtol=0,atol=1e-7)
early=early.assign_coords(x=existing.x,y=existing.y)
early["spatial_ref"]=existing["spatial_ref"]
complete=existing.processed.values==1
if len(pd.DatetimeIndex(existing.date.values[~complete]).difference(early.date.values)):
    raise RuntimeError("Existing event-period processing is unfinished.")
existing=existing.isel(date=np.flatnonzero(complete))
new_dates=pd.DatetimeIndex(early.date.values).difference(existing.date.values)
a2=xr.concat([early.sel(date=new_dates),existing],dim="date",data_vars="minimal",coords="minimal",compat="equals",join="exact").sortby("date")
a2=a2.rio.write_crs(existing.attrs["crs_wkt"])
template=a2[DNB_BAND].isel(date=0,drop=True)
ghsl=rxr.open_rasterio(GHSL_PATH,masked=True).squeeze("band",drop=True)
ghsl=ghsl.rio.reproject_match(template,resampling=Resampling.nearest).values
municipalities=gpd.read_file(MUNICIPALITIES_PATH)
region=municipalities[municipalities.PROVINCE.str.strip().str.title().isin(
    ["Samar","Eastern Samar","Northern Samar","Leyte","Southern Leyte"])].to_crs(a2.rio.crs).copy()
region["unit_name"]=region.NAME_2.str.replace("City of ","",regex=False).str.replace(" City","",regex=False).str.strip()
region["zone_id"]=np.arange(1,len(region)+1)
zone=rasterize([(g,int(i)) for g,i in zip(region.geometry,region.zone_id)],out_shape=ghsl.shape,
               transform=template.rio.transform(recalc=True),fill=0)
land=(zone>0)&np.isin(ghsl,list(GHSL_CLASS_LABELS))
land_pixels=np.flatnonzero(land)
ghsl_vector=ghsl.ravel()[land_pixels]
g3_vector=np.isin(ghsl_vector,GHSL_MASKS[SETTLEMENT_MASK])

poi_table=pd.DataFrame([
    ("Tacloban",125.0015,11.2434),("Palo",124.9904,11.1577),("Guiuan",125.7232,11.0312),
    ("Alangalang",124.8465,11.2071),("Ormoc",124.6068,11.0088),("Baybay",124.7989,10.6766)
],columns=["unit_name","longitude","latitude"])
landmarks=pd.DataFrame([
    ("Airport vicinity",125.0278,11.2275),("Port vicinity",125.0060,11.2500),
    ("Main-road vicinity",125.0030,11.2420),("Residential vicinity",125.0010,11.2650),
    ("Plaza vicinity",125.0050,11.2450),("Astrodome vicinity",125.0010,11.2240)
],columns=["unit_name","longitude","latitude"])
profiles={"Regional G3":np.flatnonzero(g3_vector)}
meta=[dict(profile="Regional G3",kind="region",class_code="G3",extent="region",color=RQ_FOUR_DAY_COLOR)]
for value,label in GHSL_CLASS_LABELS.items():
    name=f"{value} · {label}"; profiles[name]=np.flatnonzero(ghsl_vector==value)
    meta.append(dict(profile=name,kind="GHSL",class_code=str(value),extent="region",color=GHSL_CLASS_COLORS[value]))
for table,kind in [(poi_table,"centre"),(landmarks,"landmark")]:
    for point in table.itertuples():
        ix=int(abs(a2.x.values-point.longitude).argmin()); iy=int(abs(a2.y.values-point.latitude).argmin())
        for size in ([KERNEL_SIZE] if kind=="centre" else [1,3,5]):
            local=np.zeros(land.shape,dtype=bool); r=size//2
            local[max(0,iy-r):iy+r+1,max(0,ix-r):ix+r+1]=True
            if kind=="centre":
                parent_ids=region.loc[region.unit_name.eq(point.unit_name),"zone_id"].values
                local &= np.isin(zone,parent_ids)
            indices=np.flatnonzero(local.ravel()[land_pixels]&g3_vector)
            name=f"{point.unit_name} | {size}×{size}"
            profiles[name]=indices
            meta.append(dict(profile=name,kind=kind,class_code="G3",extent=f"{size}×{size}",
                color=LOCATION_COLORS.get(point.unit_name,"#174A7E")))
profile_meta=pd.DataFrame(meta)
print(f"A2: {a2.sizes['date']} available dates; {len(land_pixels):,} regional land pixels.")


In [ ]:
dates=pd.DatetimeIndex(a2.date.values)
day_blocks=np.asarray((dates-EVENT_DATE).days//AGGREGATION_DAYS)
blocks=np.arange(day_blocks.min(),day_blocks.max()+1)
cache_settings={"version":3,"stores":[str(BASELINE_STORE.resolve()),str(EXISTING_STORE.resolve())],
    "dates":dates.strftime("%Y-%m-%d").tolist(),"p95":RQ_CLIP_PERCENTILE,"reference":str(BASELINE_START),
    "minimum_reference_composites":MIN_BASELINE_OBSERVATIONS[4],"fill_values":KNOWN_FILL_VALUES,
    "aggregation_days":AGGREGATION_DAYS,"ghsl_mtime":GHSL_PATH.stat().st_mtime,
    "boundaries_mtime":MUNICIPALITIES_PATH.stat().st_mtime,
    "masks":{p:idx.tolist() for p,idx in profiles.items()}}
cache_key=hashlib.sha256(json.dumps(cache_settings,sort_keys=True).encode()).hexdigest()
cache_dir=OUTPUT_DIR/"cache"/cache_key; cache_dir.mkdir(parents=True,exist_ok=True)

def median(values,axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore",RuntimeWarning)
        return np.nanmedian(values,axis=axis)

def read_days(indices):
    batch=a2[[DNB_BAND,MQF_BAND,GAP_FILLED_BAND]].isel(date=indices).compute()
    raw=batch[DNB_BAND].values.reshape(len(indices),-1)[:,land_pixels].astype("float32")
    quality=batch[MQF_BAND].values.reshape(len(indices),-1)[:,land_pixels]
    gap=batch[GAP_FILLED_BAND].values.reshape(len(indices),-1)[:,land_pixels].astype("float32")
    raw[(quality!=0)|~np.isfinite(raw)|(raw<0)|np.isin(raw,KNOWN_FILL_VALUES)]=np.nan
    gap[~np.isfinite(gap)|(gap<0)|np.isin(gap,KNOWN_FILL_VALUES)]=np.nan
    capped=raw.copy()
    for j in range(len(indices)):
        good=raw[j,g3_vector]; good=good[np.isfinite(good)]
        if good.size: capped[j]=np.minimum(raw[j],np.percentile(good,RQ_CLIP_PERCENTILE))
    return raw,capped,gap

if not REBUILD_PROFILES and (cache_dir/"complete.json").exists():
    daily=pd.read_csv(cache_dir/"daily.csv",parse_dates=["date"])
    four_day=pd.read_csv(cache_dir/"four_day.csv",parse_dates=["date"])
    ntl0=np.load(cache_dir/"reference.npy"); reference_count=np.load(cache_dir/"reference_count.npy")
else:
    reference_cache={}; reference_composites=[]
    for block in range(-BASELINE_DAYS//AGGREGATION_DAYS,0):
        indices=np.flatnonzero(day_blocks==block)
        if len(indices):
            arrays=read_days(indices); reference_cache[block]=arrays
            reference_composites.append(median(arrays[1],axis=0))
        else: reference_composites.append(np.full(len(land_pixels),np.nan,dtype="float32"))
    reference_composites=np.asarray(reference_composites)
    reference_count=np.isfinite(reference_composites).sum(axis=0)
    ntl0=median(reference_composites,axis=0)
    reference_ok=(reference_count>=MIN_BASELINE_OBSERVATIONS[4])&np.isfinite(ntl0)&(ntl0>0)
    fixed={p:idx[reference_ok[idx]] for p,idx in profiles.items()}
    def summarize(arrays,date):
        rows=[]
        for name,idx in fixed.items():
            fresh,capped,gap=[a[idx] for a in arrays]
            valid=np.isfinite(capped); n=int(valid.sum()); sc=100*n/len(idx) if len(idx) else np.nan
            ratio=100*capped[valid].sum()/ntl0[idx][valid].sum() if n else np.nan
            rows.append((date,name,len(idx),n,sc,median(fresh),median(capped),median(gap),ratio))
        return rows
    daily_rows=[]; block_rows=[]
    for block in tqdm(blocks,desc="Observed daily and 4-day profiles"):
        indices=np.flatnonzero(day_blocks==block)
        if len(indices):
            arrays=reference_cache[block] if block in reference_cache else read_days(indices)
            for j,index in enumerate(indices): daily_rows.extend(summarize([a[j] for a in arrays],dates[index]))
            composite=[median(a,axis=0) for a in arrays]
        else: composite=[np.full(len(land_pixels),np.nan)]*3
        midpoint=EVENT_DATE+pd.Timedelta(days=int(block)*4+1.5)
        block_rows.extend(summarize(composite,midpoint))
    columns=["date","profile","fixed_pixels","valid_pixels","spatial_coverage_pct","raw_uncapped","rq_median","gap_median","reference_pct"]
    daily=pd.DataFrame(daily_rows,columns=columns); four_day=pd.DataFrame(block_rows,columns=columns)
    daily.to_csv(cache_dir/"daily.csv",index=False); four_day.to_csv(cache_dir/"four_day.csv",index=False)
    np.save(cache_dir/"reference.npy",ntl0); np.save(cache_dir/"reference_count.npy",reference_count)
    (cache_dir/"complete.json").write_text(json.dumps({"key":cache_key}))
reference_ok=(reference_count>=MIN_BASELINE_OBSERVATIONS[4])&np.isfinite(ntl0)&(ntl0>0)
fixed={p:idx[reference_ok[idx]] for p,idx in profiles.items()}
profile_meta["base_pixels"]=profile_meta.profile.map(lambda p:len(profiles[p]))
profile_meta["fixed_pixels"]=profile_meta.profile.map(lambda p:len(fixed[p]))
for frame,days in [(daily,1),(four_day,4)]:
    frame["aggregation_days"]=days
    frame["observed"]=frame.spatial_coverage_pct.ge(SPATIAL_COMPLETENESS_PCT)&frame.valid_pixels.gt(0)
    frame["raw_median_ntl"]=frame.rq_median.where(frame.observed)
    frame["normalized_ntl_pct"]=frame.reference_pct.where(frame.observed)
    frame["reference_start"],frame["reference_end"]=BASELINE_START,PRE_EVENT_END
    frame["sc_threshold_pct"],frame["p95"]=SPATIAL_COMPLETENESS_PCT,RQ_CLIP_PERCENTILE
    frame["date_start"]=frame.date-pd.to_timedelta((days-1)/2,unit="D")
    frame["date_end"]=frame.date_start+pd.to_timedelta(days-1,unit="D")
    frame["observation_status"]=np.where(frame.observed,"observed","not observable")
    frame["class_code"]=frame.profile.map(profile_meta.set_index("profile").class_code)
    frame["extent"]=frame.profile.map(profile_meta.set_index("profile").extent)
print("Fixed support — zero-pixel profiles remain visibly unavailable:")
display(profile_meta[["profile","base_pixels","fixed_pixels"]])


## 2. Spatial context and observed baseline

The first maps preserve the original GHSL classes and pre-Haiyan reference radiance. The regional and class plots preserve the existing green SC strips. A repeated radiance trough has a different meaning when SC also falls, and regional medians can change when the set of contributing pixels changes. The class heatmaps expose common gaps without treating missing blocks as zero NTL.

In [ ]:
# ------------------------------------------------------------
# Display grids — analytical values remain unchanged
# ------------------------------------------------------------

map_classes = [
    (10, "Water", "#B8D9EF"),
    *[
        (value, GHSL_CLASS_LABELS[value], GHSL_CLASS_COLORS[value])
        for value in GHSL_CLASS_LABELS
    ],
]

ghsl_grid = np.full(land.shape, np.nan)

for index, (value, _, _) in enumerate(map_classes):
    mask = (ghsl == value)
    if value != 10:
        mask &= land
    ghsl_grid[mask] = index

reference_grid = np.full(land.size, np.nan)
reference_grid[land_pixels[reference_ok]] = ntl0[reference_ok]
reference_grid = reference_grid.reshape(land.shape)

# Water is shown only where GHSL identifies water.
# Missing terrestrial NTL remains blank.
water_grid = np.where(ghsl == 10, 1.0, np.nan)

ghsl_scale = [
    [stop, color]
    for index, (_, _, color) in enumerate(map_classes)
    for stop in (
        index / len(map_classes),
        (index + 1) / len(map_classes),
    )
]

west, south, east, north = region.total_bounds
regional_bounds = (
    west - 0.03,
    south - 0.03,
    east + 0.03,
    north + 0.03,
)
tacloban_bounds = (124.94, 11.14, 125.10, 11.32)

# Dissolve internal municipality boundaries.
outline = region.dissolve().geometry.iloc[0].boundary
outline_x, outline_y = [], []

for line in (
    list(outline.geoms)
    if hasattr(outline, "geoms")
    else [outline]
):
    xx, yy = line.xy
    outline_x.extend([*xx, None])
    outline_y.extend([*yy, None])


# ------------------------------------------------------------
# Two large regional maps; Tacloban insets at their top right
# ------------------------------------------------------------

fig = go.Figure()
style_figure(fig, "", 1700, 1050)

panels = [
    # axis, grid, bounds, x-domain, y-domain, inset
    (1, ghsl_grid, regional_bounds, [0.015, 0.345], [0.22, 0.95], False),
    (2, reference_grid, regional_bounds, [0.515, 0.845], [0.22, 0.95], False),
    (3, ghsl_grid, tacloban_bounds, [0.305, 0.490], [0.62, 0.925], True),
    (4, reference_grid, tacloban_bounds, [0.805, 0.990], [0.62, 0.925], True),
]

for number, grid, bounds, x_domain, y_domain, inset in panels:
    is_ghsl = number in (1, 3)
    xref = "x" if number == 1 else f"x{number}"
    yref = "y" if number == 1 else f"y{number}"
    xaxis_name = "xaxis" if number == 1 else f"xaxis{number}"
    yaxis_name = "yaxis" if number == 1 else f"yaxis{number}"

    xi = np.flatnonzero(
        (a2.x.values >= bounds[0])
        & (a2.x.values <= bounds[2])
    )
    yi = np.flatnonzero(
        (a2.y.values >= bounds[1])
        & (a2.y.values <= bounds[3])
    )

    xx = a2.x.values[xi]
    yy = a2.y.values[yi]

    if not is_ghsl:
        fig.add_trace(go.Heatmap(
            x=xx,
            y=yy,
            z=water_grid[np.ix_(yi, xi)],
            xaxis=xref,
            yaxis=yref,
            colorscale=[[0, "#B8D9EF"], [1, "#B8D9EF"]],
            zmin=0,
            zmax=1,
            showscale=False,
            hoverinfo="skip",
        ))

    fig.add_trace(go.Heatmap(
        x=xx,
        y=yy,
        z=grid[np.ix_(yi, xi)],
        xaxis=xref,
        yaxis=yref,
        colorscale=ghsl_scale if is_ghsl else "Inferno",
        zmin=-0.5 if is_ghsl else NTL_DISPLAY_MIN,
        zmax=len(map_classes) - 0.5 if is_ghsl else NTL_DISPLAY_MAX,
        showscale=(number == 2),
        zsmooth=False,
        hoverongaps=False,
        hoverinfo="skip" if is_ghsl else None,
        hovertemplate=(
            None if is_ghsl else
            "%{y:.4f}°N, %{x:.4f}°E"
            "<br>NTL: %{z:.3f} nW cm⁻² sr⁻¹<extra></extra>"
        ),
        colorbar=dict(
            orientation="h",
            title=dict(
                text="DNB-BRDF nighttime lights (nW cm⁻² sr⁻¹)",
                side="top",
            ),
            x=0.735,
            xanchor="center",
            y=0.11,
            yanchor="middle",
            len=0.36,
            thickness=18,
            outlinewidth=0,
            tickvals=np.linspace(NTL_DISPLAY_MIN, NTL_DISPLAY_MAX, 6),
        ),
    ))

    # Outer outline only: fine enough to preserve raster visibility.
    fig.add_trace(go.Scatter(
        x=outline_x,
        y=outline_y,
        xaxis=xref,
        yaxis=yref,
        mode="lines",
        line=dict(color="rgba(36,59,90,0.28)", width=0.4),
        showlegend=False,
        hoverinfo="skip",
    ))

    fig.update_layout({
        xaxis_name: dict(
            domain=x_domain,
            anchor=yref,
            range=[bounds[0], bounds[2]],
            constrain="domain",
            showgrid=False,
            zeroline=False,
            showticklabels=not inset,
            ticks="" if inset else "outside",
            tickvals=np.arange(
                np.ceil(bounds[0]), np.floor(bounds[2]) + 1, 1
            ),
            tickformat=".0f",
            ticksuffix="°E",
            title=None,
        ),
        yaxis_name: dict(
            domain=y_domain,
            anchor=xref,
            range=[bounds[1], bounds[3]],
            scaleanchor=xref,
            scaleratio=1 / np.cos(np.deg2rad(
                (bounds[1] + bounds[3]) / 2
            )),
            constrain="domain",
            showgrid=False,
            zeroline=False,
            showticklabels=not inset,
            ticks="" if inset else "outside",
            tickvals=np.arange(
                np.ceil(bounds[1]), np.floor(bounds[3]) + 1, 1
            ),
            tickformat=".0f",
            ticksuffix="°N",
            title=None,
        ),
    })

    if inset:
        # Frame and labels only inside the inset.
        fig.add_shape(
            type="rect",
            xref=xref,
            yref=yref,
            x0=bounds[0],
            x1=bounds[2],
            y0=bounds[1],
            y1=bounds[3],
            line=dict(color=PLOT_TEXT_COLOR, width=1),
            fillcolor="rgba(0,0,0,0)",
        )

        visible = poi_table[
            poi_table.longitude.between(bounds[0], bounds[2])
            & poi_table.latitude.between(bounds[1], bounds[3])
        ]

        fig.add_trace(go.Scatter(
            x=visible.longitude,
            y=visible.latitude,
            text=visible.unit_name,
            xaxis=xref,
            yaxis=yref,
            mode="markers+text",
            marker=dict(
                color="white",
                size=5,
                line=dict(color=PLOT_TEXT_COLOR, width=0.8),
            ),
            textposition="top center",
            textfont=dict(size=11, color=PLOT_TEXT_COLOR),
            showlegend=False,
        ))

    else:
        # Locate the inset footprint without six overlapping regional labels.
        fig.add_shape(
            type="rect",
            xref=xref,
            yref=yref,
            x0=tacloban_bounds[0],
            x1=tacloban_bounds[2],
            y0=tacloban_bounds[1],
            y1=tacloban_bounds[3],
            line=dict(color="#64748B", width=1, dash="dot"),
            fillcolor="rgba(0,0,0,0)",
        )


# ------------------------------------------------------------
# Panel headings — no main title
# ------------------------------------------------------------

for x, text in [
    (0.18, "GHSL settlement context"),
    (0.68, "60-day pixel baseline"),
]:
    fig.add_annotation(
        x=x, y=0.995,
        xref="paper", yref="paper",
        text=f"{text}",
        showarrow=False,
        font=dict(size=19, color=PLOT_TEXT_COLOR),
    )

for x in (0.3975, 0.8975):
    fig.add_annotation(
        x=x, y=0.945,
        xref="paper", yref="paper",
        text="Tacloban",
        showarrow=False,
        font=dict(size=14, color=PLOT_TEXT_COLOR),
    )


# ------------------------------------------------------------
# Fixed GHSL legend: two columns × four rows, including water
# ------------------------------------------------------------

for index, (value, label, color) in enumerate(map_classes):
    column, row = divmod(index, 4)
    x = 0.055 + column * 0.235
    y = 0.165 - row * 0.042

    fig.add_shape(
        type="rect",
        xref="paper", yref="paper",
        x0=x, x1=x + 0.008,
        y0=y - 0.006, y1=y + 0.006,
        fillcolor=color,
        line=dict(width=0),
    )
    fig.add_annotation(
        x=x + 0.013, y=y,
        xref="paper", yref="paper",
        text=f"{value} · {label}",
        showarrow=False,
        xanchor="left",
        yanchor="middle",
        font=dict(size=17, color=PLOT_TEXT_COLOR),
    )

fig.update_layout(
    title_text="",
    showlegend=False,
    margin=dict(l=55, r=35, t=35, b=25),
)

finish_figure(fig, "01_context_and_reference")

In [ ]:

display_start = CHARACTERIZATION_START.to_period("M").to_timestamp()
month_starts = pd.date_range(
    display_start, CHARACTERIZATION_END, freq="MS"
)
quarter_ticks = pd.date_range(
    display_start, CHARACTERIZATION_END, freq="3MS"
)


In [ ]:
calendar = pd.date_range(dates.min(), dates.max(), freq="D")
regional = (
    daily.loc[daily.profile.eq("Regional G3")]
    .set_index("date").reindex(calendar)
)
regional4 = (
    four_day.loc[four_day.profile.eq("Regional G3")]
    .sort_values("date")
)

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[.09, .91], vertical_spacing=.06,
)
style_figure(fig, "", 1600, 500)

fig.add_trace(go.Heatmap(
    x=calendar, y=["Daily SC"], z=[regional.spatial_coverage_pct],
    colorscale=SC_GREEN_COLORSCALE, zmin=0, zmax=100,
    colorbar=dict(
        title=dict(text="SC (%)", font=dict(size=12)),
        tickfont=dict(size=12), len=.2, x = 0.95, y=.98, thickness=15,
    ),
    hovertemplate="%{x|%d %b %Y}<br>Daily SC: %{z:.1f}%<extra></extra>",
    hoverongaps=False,
), row=1, col=1)

# Draw uncapped observations first, behind the summaries.
fig.add_trace(go.Scatter(
    x=calendar, y=regional.raw_uncapped,
    mode="markers", name="Daily DNB-BRDF NTL",
    marker=dict(size=5, color='gray', opacity=.50),
    hovertemplate="%{x|%d %b %Y}<br>Daily DNB: %{y:.4f}<extra></extra>",
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=calendar, y=regional.gap_median,
    mode="lines", name="Gap-filled",
    line=dict(color=GAP_FILLED_COLOR, width=1.2),
    opacity=.55, connectgaps=False,
), row=2, col=1)


fig.add_trace(go.Scatter(
    x=regional4.date, y=regional4.raw_median_ntl,
    mode="lines", name="Reliability-Qualified NTL (4D) Median",
    line=dict(color='black', width=2.5, shape="hv"),
    marker=dict(size=5, line=dict(color="white", width=.6)),
    connectgaps=False,
    customdata=regional4[["spatial_coverage_pct", "valid_pixels"]],
    hovertemplate=(
        "%{x|%d %b %Y}<br>Four-day NTL: %{y:.4f}"
        "<br>Four-day SC: %{customdata[0]:.1f}%"
        "<br>Valid pixels: %{customdata[1]:.0f}<extra></extra>"
    ),
), row=2, col=1)

for row in (1, 2):
    add_haiyan_marker(fig, row, 1)

fig.update_layout(
    title=None, font=dict(size=17),
    margin=dict(l=40, r=40, t=40, b=40),
    legend=dict(
        orientation="h", x=0.45, y=0.94,
        xanchor="left", yanchor="top", font=dict(size=20),
    ),
)
fig.update_xaxes(
    tickfont=dict(size=16), tickformat="%b<br>%Y",
    dtick="M6", title_text=None,
)
fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_yaxes(
    showticklabels=False, showgrid=False, title_text=None,
    row=1, col=1,
)

fig.update_yaxes(
    range=[0, 1.2], dtick=.3, tickfont=dict(size=17),
    title=dict(text="NTL (nW cm⁻² sr⁻¹)", font=dict(size=19)),
    row=2, col=1
)


# fig.update_yaxes(
#     title=dict(text="NTL (nW cm⁻² sr⁻¹)", font=dict(size=20)),
#     row=2, col=1, type="log"
# )

month_starts = pd.date_range(
    display_start, CHARACTERIZATION_END, freq="MS"
)

for month in month_starts:
    fig.add_vline(
        x=month,
        line_color=(
            "rgba(36,59,90,.20)"
            if month.month == 1
            else "rgba(36,59,90,.10)"
        ),
        line_width=1.2 if month.month == 1 else .8,
        layer="below",
        row=row, col=1,
    )

finish_figure(fig, "02_regional_observations")

In [ ]:
first_month = pd.Timestamp(CHARACTERIZATION_START).to_period("M").to_timestamp()
month_starts = pd.date_range(first_month, CHARACTERIZATION_END, freq="MS")
half_year_ticks = pd.date_range(first_month, CHARACTERIZATION_END, freq="6MS")

RAW_DOT_COLOR = "#87919B"


def baseline_panels(names, colors, ncols, filename, show=True):
    nrows = int(np.ceil(len(names) / ncols))
    fig = make_subplots(
        rows=2*nrows, cols=ncols,
        row_heights=[.07, .43]*nrows,
        vertical_spacing=.09,
        horizontal_spacing=.055,
    )
    style_figure(fig, "", 1800 if ncols == 4 else 1600, 800)

    for i, (name, color) in enumerate(zip(names, colors)):
        col = i % ncols + 1
        sc_row = 2*(i // ncols) + 1
        ntl_row = sc_row + 1

        d = (
            daily.loc[daily.profile.eq(name)]
            .set_index("date")
            .reindex(calendar)
            .loc[CHARACTERIZATION_START:CHARACTERIZATION_END]
        )
        g = four_day.loc[
            four_day.profile.eq(name)
            & four_day.date_start.ge(CHARACTERIZATION_START)
            & four_day.date_end.le(CHARACTERIZATION_END)
        ].sort_values("date")

        fig.add_trace(go.Heatmap(
            x=g.date, y=["SC"], z=[g.spatial_coverage_pct],
            colorscale=SC_GREEN_COLORSCALE,
            zmin=0, zmax=100,
            showscale=(i == 0),
            colorbar=dict(
                title=dict(text="SC (%)", font=dict(size=12)),
                tickfont=dict(size=12), len=.1, x = 1, y=.99, thickness=15,
            ),
            hovertemplate=(
                "%{x|%d %b %Y}<br>Four-day SC: %{z:.1f}%"
                "<extra></extra>"
            ),
            hoverongaps=False,
        ), row=sc_row, col=col)

        # Gray daily observations sit behind the RQ line.
        fig.add_trace(go.Scatter(
            x=d.index, y=d.raw_uncapped,
            mode="markers",
            marker=dict(size=3.5, color=RAW_DOT_COLOR, opacity=.35),
            showlegend=False,
            hovertemplate=(
                "%{x|%d %b %Y}<br>Uncapped daily NTL: %{y:.4f}"
                "<extra></extra>"
            ),
        ), row=ntl_row, col=col)

        legend_id = "legend" if i == 0 else f"legend{i+1}"
        fig.add_trace(go.Scatter(
            x=g.date, y=g.raw_median_ntl,
            mode="lines",
            name=name,
            legend=legend_id,
            showlegend=True,
            line=dict(color=color, width=1.6, shape="hv"),
            connectgaps=False,
            customdata=g[["spatial_coverage_pct", "valid_pixels"]],
            hovertemplate=(
                "%{x|%d %b %Y}<br>Four-day RQ NTL: %{y:.4f}"
                "<br>SC: %{customdata[0]:.1f}%"
                "<br>Valid pixels: %{customdata[1]:.0f}"
                "<extra></extra>"
            ),
        ), row=ntl_row, col=col)

        # Class legend sits between its SC strip and NTL panel.
        axis_number = (ntl_row-1)*ncols + col
        suffix = "" if axis_number == 1 else str(axis_number)
        xdomain = fig.layout[f"xaxis{suffix}"].domain
        ydomain = fig.layout[f"yaxis{suffix}"].domain
        fig.update_layout(**{
            legend_id: dict(
                x=xdomain[0],
                y=ydomain[1] + .007,
                xanchor="left",
                yanchor="bottom",
                font=dict(size=16),
                bgcolor="rgba(0,0,0,0)",
                borderwidth=0,
            )
        })

        for row in (sc_row, ntl_row):
            fig.update_xaxes(
                range=[CHARACTERIZATION_START, CHARACTERIZATION_END],
                tickvals=half_year_ticks,
                tickformat="%b %Y",
                tickfont=dict(size=15),
                showticklabels=(row == ntl_row),
                title_text=None,
                showgrid=False,
                row=row, col=col,
            )

        fig.update_yaxes(
            showticklabels=False,
            showgrid=False,
            title_text=None,
            row=sc_row, col=col,
        )

        is_centre = name in profile_meta.loc[
            profile_meta.kind.eq("centre"), "profile"
        ].values

        if is_centre:
            values = np.concatenate([
                d.raw_uncapped.to_numpy(dtype=float),
                g.raw_median_ntl.to_numpy(dtype=float),
            ])
            values = values[np.isfinite(values)]
            ymax = (
                max(0.5, np.ceil(values.max()*1.05*2)/2)
                if values.size else 1.2
            )
        else:
            ymax = 5 if name.startswith("30 ·") else 1.2

        fig.update_yaxes(
            range=[0, ymax],
            tickmode="auto",
            nticks=5,
            tickfont=dict(size=15),
            title=dict(
                text="NTL (nW cm⁻² sr⁻¹)" if col == 1 else "",
                font=dict(size=17),
            ),
            row=ntl_row, col=col,
        )

    fig.update_layout(
        title=None,
        font=dict(size=16),
        margin=dict(l=95, r=115, t=45, b=65),
    )

    if show:
        finish_figure(fig, filename)
    return fig


# Seven GHSL classes; panel eight compares them on one axis.
class_names = [
    f"{code} · {label}"
    for code, label in GHSL_CLASS_LABELS.items()
]
class_colors = [
    GHSL_CLASS_COLORS[code]
    for code in GHSL_CLASS_LABELS
]

fig = baseline_panels(
    class_names, class_colors, 4,
    "03_ghsl_baseline", show=False,
)

coverage_rows = []

for code, name, color in zip(
    GHSL_CLASS_LABELS, class_names, class_colors
):
    d = (
        daily.loc[daily.profile.eq(name)]
        .set_index("date")
        .reindex(calendar)
        .loc[CHARACTERIZATION_START:CHARACTERIZATION_END]
    )
    g = four_day.loc[
        four_day.profile.eq(name)
        & four_day.date_start.ge(CHARACTERIZATION_START)
        & four_day.date_end.le(CHARACTERIZATION_END)
    ].sort_values("date")

    coverage_rows.append(
        g.set_index("date").spatial_coverage_pct.rename(str(code))
    )

    fig.add_trace(go.Scatter(
        x=d.index, y=d.raw_uncapped,
        mode="markers",
        marker=dict(size=3, color=RAW_DOT_COLOR, opacity=.13),
        showlegend=False,
        hoverinfo="skip",
    ), row=4, col=4)

    fig.add_trace(go.Scatter(
        x=g.date, y=g.raw_median_ntl,
        mode="lines",
        line=dict(color=color, width=1.6, shape="hv"),
        name=name,
        showlegend=False,
        connectgaps=False,
        hovertemplate=(
            f"{name}<br>"
            "%{x|%d %b %Y}<br>Four-day RQ NTL: %{y:.4f}"
            "<extra></extra>"
        ),
    ), row=4, col=4)

# Seven separate SC rows. Missing coverage remains missing.
coverage = pd.concat(coverage_rows, axis=1).sort_index()
fig.add_trace(go.Heatmap(
    x=coverage.index,
    y=coverage.columns.tolist(),
    z=coverage.T.values,
    colorscale=SC_GREEN_COLORSCALE,
    zmin=0, zmax=100,
    showscale=False,
    hoverongaps=False,
    hovertemplate=(
        "GHSL %{y}<br>%{x|%d %b %Y}"
        "<br>Four-day SC: %{z:.1f}%<extra></extra>"
    ),
), row=3, col=4)

# Match the date labels in the comparison panel.
for row in (3, 4):
    fig.update_xaxes(
        range=[CHARACTERIZATION_START, CHARACTERIZATION_END],
        tickvals=half_year_ticks,
        tickformat="%b %Y",
        tickfont=dict(size=15),
        showticklabels=(row == 4),
        title_text=None,
        showgrid=False,
        row=row, col=4,
    )

# Faint monthly guides on every NTL panel.
for ntl_row in (2, 4):
    for col in range(1, 5):
        for month in month_starts:
            fig.add_vline(
                x=month,
                line_color=(
                    "rgba(36,59,90,.15)"
                    if month.month == 1
                    else "rgba(36,59,90,.08)"
                ),
                line_width=1 if month.month == 1 else .7,
                layer="below",
                row=ntl_row, col=col,
            )

fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    title_text=None,
    row=3, col=4,
)
fig.update_yaxes(
    range=[0, 5],
    dtick=1,
    tickfont=dict(size=15),
    title_text=None,
    row=4, col=4,
)

finish_figure(fig, "03_ghsl_baseline")

In [ ]:
centre_meta = profile_meta.loc[
    profile_meta.kind.eq("centre")
]

fig = baseline_panels(
    names=centre_meta.profile.tolist(),
    colors=centre_meta.color.tolist(),
    ncols=3,
    filename="04_six_centres",
    show=False,
)

for name in centre_meta.profile:
    g = four_day.loc[
        four_day.profile.eq(name)
        & four_day.date_start.ge(CHARACTERIZATION_START)
        & four_day.date_end.le(CHARACTERIZATION_END)
    ].sort_values("date")

    # Fresh-pixel four-day median, without the regional P95 cap.
    # Keep the existing spatial-completeness qualification.
    uncapped_4d = g.raw_uncapped.where(g.observed)

    for trace in fig.data:
        if trace.name == name:
            trace.y = uncapped_4d.to_numpy()
            trace.hovertemplate = (
                "%{x|%d %b %Y}"
                "<br>Uncapped four-day NTL: %{y:.2f}"
                "<br>SC: %{customdata[0]:.1f}%"
                "<br>Valid pixels: %{customdata[1]:.0f}"
                "<extra></extra>"
            )
            break

finish_figure(fig, "04_six_centres")

In [ ]:
pre = four_day.loc[
    four_day["date_start"].ge(CHARACTERIZATION_START)
    & four_day["date_end"].le(CHARACTERIZATION_END)
].copy()

In [ ]:
class_names = [
    f"{code} · {label}"
    for code, label in GHSL_CLASS_LABELS.items()
]

coverage = (
    pre.loc[pre.profile.isin(class_names)]
    .pivot(index="profile", columns="date", values="spatial_coverage_pct")
    .reindex(class_names)
)

fig = go.Figure(go.Heatmap(
    x=coverage.columns,
    y=coverage.index,
    z=coverage.values,
    colorscale=SC_GREEN_COLORSCALE,
    zmin=0, zmax=100,
    hoverongaps=False,
    colorbar=dict(
        title=dict(text="SC (%)", font=dict(size=18)),
        tickfont=dict(size=16),
        thickness=18,
    ),
    hovertemplate=(
        "%{y}<br>%{x|%d %b %Y}"
        "<br>Four-day SC: %{z:.1f}%<extra></extra>"
    ),
))

style_figure(fig, "", 1600, 300)
fig.update_layout(
    title=None,
    font=dict(size=17),
    margin=dict(l=230, r=90, t=25, b=65),
)

fig.update_xaxes(
    tickformat="%b %Y",
    dtick="M3",
    tick0="2012-01-01",
    tickfont=dict(size=15),
    showgrid=False,
)

month_starts = pd.date_range(
    coverage.columns.min().to_period("M").to_timestamp(),
    coverage.columns.max().to_period("M").to_timestamp(),
    freq="MS",
)

for month in month_starts:
    date = month.strftime("%Y-%m-%d")
    fig.add_shape(
        type="line",
        xref="x", yref="paper",
        x0=date, x1=date,
        y0=0, y1=1,
        line=dict(color="black", width=1),
        layer="above",
    )

fig.update_yaxes(
    autorange="reversed",
    tickfont=dict(size=16), title_text=None,
    showgrid=False,
)
finish_figure(fig, "07_four_day_availability")

In [ ]:
levels = (
    pre.loc[pre.profile.isin(class_names)]
    .pivot(index="profile", columns="date", values="raw_median_ntl")
    .reindex(class_names)
)
sc = coverage.reindex(index=levels.index, columns=levels.columns)

# Log colour display only; hover reports the original radiance.
display_values = np.log1p(
    np.clip(levels.values, 0, NTL_DISPLAY_MAX)
)
hover_values = np.stack([levels.values, sc.values], axis=-1)

radiance_ticks = [0, .2, .5, 1, 2, 5]

fig = go.Figure(go.Heatmap(
    x=levels.columns,
    y=levels.index,
    z=display_values,
    customdata=hover_values,
    colorscale="Inferno",
    zmin=0,
    zmax=np.log1p(NTL_DISPLAY_MAX),
    hoverongaps=False,
    colorbar=dict(
        title=dict(text="NTL", font=dict(size=18)),
        tickvals=np.log1p(radiance_ticks),
        ticktext=[str(value) for value in radiance_ticks],
        tickfont=dict(size=16),
        thickness=18,
    ),
    hovertemplate=(
        "%{y}<br>%{x|%d %b %Y}"
        "<br>RQ NTL: %{customdata[0]:.3f} nW cm⁻² sr⁻¹"
        "<br>SC: %{customdata[1]:.1f}%<extra></extra>"
    ),
))

style_figure(fig, "", 1600, 300)
fig.update_layout(
    title=None,
    font=dict(size=17),
    margin=dict(l=230, r=90, t=25, b=65),
)
fig.update_xaxes(
    tickformat="%b %Y",
    dtick="M3",
    tick0="2012-01-01",
    tickfont=dict(size=15),
    showgrid=False,
)

month_starts = pd.date_range(
    coverage.columns.min().to_period("M").to_timestamp(),
    coverage.columns.max().to_period("M").to_timestamp(),
    freq="MS",
)

for month in month_starts:
    date = month.strftime("%Y-%m-%d")
    fig.add_shape(
        type="line",
        xref="x", yref="paper",
        x0=date, x1=date,
        y0=0, y1=1,
        line=dict(color="black", width=1),
        layer="above",
    )

fig.update_yaxes(
    autorange="reversed",
    tickfont=dict(size=16), title_text=None,
    showgrid=False,
)
finish_figure(fig, "08_four_day_levels")

## 3. Brightness, temporal MAD and the processing stage

The retained three-panel pixel figure compares source DNB-BRDF, daily reliability qualification, and four-day RQ NTL on the same baseline period. Each dot is a pixel, not a class average. A lower MAD among bright pixels is a descriptive pattern; it does not by itself establish higher measurement quality. Check how the relationship changes after MQF filtering, the regional P95 cap, the SC gate and four-day compositing.

In [ ]:
# Same pre-Haiyan pixels, three processing stages:
# 1. Source DNB-BRDF: no MQF, P95, or SC gate
# 2. Daily RQ: MQF == 0, daily G3 P95 cap, regional daily SC gate
# 3. Four-day RQ: median of capped daily pixels, regional 4D SC gate

baseline_calendar = pd.date_range(
    CHARACTERIZATION_START, PRE_EVENT_END, freq="D"
)
pre_idx = np.flatnonzero(
    (dates >= CHARACTERIZATION_START) & (dates <= PRE_EVENT_END)
)
pre_dates = dates[pre_idx]

# Consider all regional land pixels. The common-support rule below
# determines which pixels appear in every comparison panel.
pixel_columns = np.arange(len(land_pixels))
flat_pixels = land_pixels[pixel_columns]
pixel_classes = ghsl_vector[pixel_columns]
n_pixels = len(pixel_columns)

def pixel_median_and_mad(values):
    n_observed = np.isfinite(values).sum(axis=0)
    centre = np.full(values.shape[1], np.nan)
    spread = np.full(values.shape[1], np.nan)

    for start in range(0, values.shape[1], 5000):
        sl = slice(start, start + 5000)
        part = values[:, sl]
        centre[sl] = median(part, axis=0)
        spread[sl] = median(np.abs(part - centre[sl]), axis=0)

    return centre, spread, n_observed

daily_values = np.full(
    (len(pre_idx), n_pixels), np.nan, dtype=np.float32
)

# Stage 1: source DNB-BRDF, without quality or SC filtering.
for start in tqdm(range(0, len(pre_idx), 24), desc="Source DNB-BRDF"):
    batch_idx = pre_idx[start:start + 24]
    values = (
        a2[DNB_BAND].isel(date=batch_idx).compute().values
        .reshape(len(batch_idx), -1)[:, flat_pixels]
        .astype(np.float32)
    )
    values[
        ~np.isfinite(values)
        | (values < 0)
        | np.isin(values, KNOWN_FILL_VALUES)
    ] = np.nan
    daily_values[start:start + len(batch_idx)] = values

raw_median, raw_mad, raw_n = pixel_median_and_mad(daily_values)

# Replace source values with MQF-qualified, P95-capped daily pixels.
# Keep all dates for now: 4D composites are formed before the 4D SC gate.
for start in tqdm(range(0, len(pre_idx), 24), desc="Qualified DNB-BRDF"):
    batch_idx = pre_idx[start:start + 24]
    daily_values[start:start + len(batch_idx)] = (
        read_days(batch_idx)[1][:, pixel_columns]
    )

regional4 = pd.read_csv(
    cache_dir / "four_day.csv",
    usecols=[
        "date", "profile", "valid_pixels",
        "spatial_coverage_pct",
    ],
    parse_dates=["date"],
)
regional4 = regional4.loc[
    regional4["profile"].eq("Regional G3")
].copy()
regional4["date_start"] = regional4["date"] - pd.Timedelta(days=1.5)
regional4["date_end"] = regional4["date"] + pd.Timedelta(days=1.5)
regional4["observed"] = (
    regional4["spatial_coverage_pct"].ge(
        SPATIAL_COMPLETENESS_PCT
    )
    & regional4["valid_pixels"].gt(0)
)
regional4 = regional4.loc[
    regional4["date_start"].ge(CHARACTERIZATION_START)
    & regional4["date_end"].le(PRE_EVENT_END)
].sort_values("date")

# Stage 3: four-day pixel composites on qualified regional blocks.
block_values = np.full(
    (len(regional4), n_pixels), np.nan, dtype=np.float32
)
pre_block_ids = day_blocks[pre_idx]

for j, block in enumerate(regional4.itertuples()):
    if not block.observed:
        continue

    block_id = int(
        (block.date_start - EVENT_DATE).days // AGGREGATION_DAYS
    )
    rows = np.flatnonzero(pre_block_ids == block_id)

    if len(rows):
        block_values[j] = median(daily_values[rows], axis=0)

four_median, four_mad, four_n = pixel_median_and_mad(block_values)

# Stage 2: apply the daily regional SC gate to daily RQ pixels.
regional_daily = pd.read_csv(
    cache_dir / "daily.csv",
    usecols=[
        "date", "profile", "valid_pixels",
        "spatial_coverage_pct",
    ],
    parse_dates=["date"],
)
regional_daily = regional_daily.loc[
    regional_daily["profile"].eq("Regional G3")
    & regional_daily["date"].between(
        CHARACTERIZATION_START, PRE_EVENT_END
    )
]
qualified_dates = pd.DatetimeIndex(
    regional_daily.loc[
        regional_daily["spatial_coverage_pct"].ge(
            SPATIAL_COMPLETENESS_PCT
        )
        & regional_daily["valid_pixels"].gt(0),
        "date",
    ]
)
daily_values[~pre_dates.isin(qualified_dates)] = np.nan

rq_median, rq_mad, rq_n = pixel_median_and_mad(daily_values)
del daily_values, block_values

# Require the same pixel to have sufficient support in all three stages.
minimum_daily = max(
    3, int(np.ceil(.10 * len(baseline_calendar)))
)
minimum_four = max(
    3, int(np.ceil(.10 * len(regional4)))
)

common = (
    (raw_n >= minimum_daily)
    & (rq_n >= minimum_daily)
    & (four_n >= minimum_four)
    & np.isfinite(raw_median) & np.isfinite(raw_mad)
    & np.isfinite(rq_median) & np.isfinite(rq_mad)
    & np.isfinite(four_median) & np.isfinite(four_mad)
)

In [ ]:
stages = [
    ("Source DNB-BRDF",
     raw_median, raw_mad, raw_n, len(baseline_calendar)),
    ("Daily RQ NTL",
     rq_median, rq_mad, rq_n, len(baseline_calendar)),
    ("RQ NTL (4D)",
     four_median, four_mad, four_n, len(regional4)),
]

fig = make_subplots(
    rows=1, cols=3,
    horizontal_spacing=.055,
    subplot_titles=[stage[0] for stage in stages],
)

for col, (_, levels, mads, counts, denominator) in enumerate(stages, 1):
    for code, label in GHSL_CLASS_LABELS.items():
        selected = common & (pixel_classes == code)

        fig.add_trace(go.Scattergl(
            x=levels[selected],
            y=mads[selected],
            mode="markers",
            name=f"{code} · {label}",
            legendgroup=str(code),
            showlegend=(col == 1),
            marker=dict(
                size=6,
                color=GHSL_CLASS_COLORS[code],
                opacity=.30,
            ),
            customdata=np.column_stack([
                counts[selected],
                100 * counts[selected] / denominator,
            ]),
            hovertemplate=(
                f"GHSL {code} · {label}"
                "<br>Pixel median: %{x:.2f}"
                "<br>Temporal MAD: %{y:.2f}"
                "<br>Observed: %{customdata[0]:.0f}"
                " (%{customdata[1]:.1f}%)"
                "<extra></extra>"
            ),
        ), row=1, col=col)

    fig.update_xaxes(
        title_text="Pixel median (nW cm⁻² sr⁻¹)",
        rangemode="tozero",
        tickfont=dict(size=15),
        title_font=dict(size=16),
        row=1, col=col,
    )
    fig.update_yaxes(
        title_text="Temporal MAD (nW cm⁻² sr⁻¹)" if col == 1 else "",
        rangemode="tozero",
        tickfont=dict(size=15),
        title_font=dict(size=16),
        row=1, col=col,
    )

style_figure(fig, "", 1200, 600)
fig.update_layout(
    title=None,
    margin=dict(l=100, r=45, t=75, b=165),
    legend=dict(
        orientation="h",
        x=0, y=-.20,
        font=dict(size=15),
    ),
)
fig.update_annotations(font=dict(size=18, color=PLOT_TEXT_COLOR))
finish_figure(fig, "pixel_median_mad_three_processing_stages")

print(
    f"{common.sum():,} identical pixels shown in each panel; "
    f"daily baseline: {CHARACTERIZATION_START:%d %b %Y}–"
    f"{PRE_EVENT_END:%d %b %Y}."
)

## 4. When the regional daily median approaches zero

The original checks distinguish an exact zero from a value near the product floor. The new paired maps inspect one low-radiance day, the **reference brightness of the very pixels observed on that day**, and a comparison day with similar SC. This separates a low measured value from a dark-biased contributing-pixel set. The maps are display-capped at 5 nW cm⁻² sr⁻¹; table values are not capped for display. A single case is diagnostic, not a regional causal conclusion.

In [ ]:
zero_daily = daily.loc[
    daily.profile.eq("Regional G3") & daily.raw_uncapped.eq(0),
    ["date", "valid_pixels", "fixed_pixels",
     "spatial_coverage_pct", "observed", "gap_median"]
].sort_values("spatial_coverage_pct", ascending=False)

zero_4d = four_day.loc[
    four_day.profile.eq("Regional G3") & four_day.raw_median_ntl.eq(0),
    ["date", "valid_pixels", "fixed_pixels", "spatial_coverage_pct"]
]

print(
    f"Daily zero medians: {len(zero_daily)}; "
    f"passing the {SPATIAL_COMPLETENESS_PCT:g}% SC gate: "
    f"{zero_daily.observed.sum()}"
)
print(f"Four-day RQ zero medians: {len(zero_4d)}")
display(zero_daily.head(20))

In [ ]:
regional_check = daily.loc[
    daily.profile.eq("Regional G3")
].sort_values("date").copy()

# Diagnostic threshold only; it does not alter the RQ series.
regional_check["near_zero"] = regional_check.raw_uncapped.le(0.05)
regional_check["SC_band"] = pd.cut(
    regional_check.spatial_coverage_pct,
    bins=[0, 10, 25, 50, 75, 101],
    labels=["<10%", "10–25%", "25–50%", "50–75%", "≥75%"],
    right=False,
)

by_coverage = regional_check.groupby(
    "SC_band", observed=False
).agg(
    days=("date", "size"),
    near_zero_days=("near_zero", "sum"),
)
by_coverage["near_zero_share_pct"] = (
    100 * by_coverage.near_zero_days / by_coverage.days
)
display(by_coverage.round(1))

near_zero_days = regional_check.loc[
    regional_check.near_zero,
    ["date", "raw_uncapped", "gap_median",
     "spatial_coverage_pct", "valid_pixels", "observed"]
].copy()
near_zero_days["days_since_previous"] = (
    near_zero_days.date.diff().dt.days
)
display(near_zero_days)

In [ ]:
regional_pre_daily = daily.loc[
    daily.profile.eq("Regional G3")
    & daily.date.between(CHARACTERIZATION_START, PRE_EVENT_END)
    & daily.observed
].sort_values("date").copy()
low_candidates = regional_pre_daily.loc[
    regional_pre_daily.raw_uncapped.le(.05)
].sort_values(["raw_uncapped", "spatial_coverage_pct"], ascending=[True, False])
if low_candidates.empty:
    low_candidates = regional_pre_daily.nsmallest(1, "raw_uncapped")
low_day = low_candidates.iloc[0]
high_candidates = regional_pre_daily.loc[
    regional_pre_daily.raw_uncapped.ge(regional_pre_daily.raw_uncapped.quantile(.75))
    & regional_pre_daily.date.sub(low_day.date).abs().le(pd.Timedelta(days=60))
].copy()
if high_candidates.empty:
    high_candidates = regional_pre_daily.loc[
        regional_pre_daily.raw_uncapped.ge(regional_pre_daily.raw_uncapped.quantile(.75))
    ].copy()
high_candidates["match_cost"] = (
    (high_candidates.spatial_coverage_pct - low_day.spatial_coverage_pct).abs()
    + .05 * high_candidates.date.sub(low_day.date).abs().dt.days
)
comparison_day = high_candidates.nsmallest(1, "match_cost").iloc[0]

case_dates = pd.DatetimeIndex([low_day.date, comparison_day.date])
case_raw, _, _ = read_days(dates.get_indexer(case_dates))
support_idx = fixed["Regional G3"]
case_rows, case_maps = [], []
for j, date in enumerate(case_dates):
    values = case_raw[j, support_idx]
    valid = np.isfinite(values)
    baseline_same_pixels = ntl0[support_idx][valid]
    case_rows.append({
        "case": "Low day" if j == 0 else "SC-matched brighter day",
        "date": date, "SC_pct": 100 * valid.mean(),
        "observed_pixels": int(valid.sum()),
        "observed_median": median(values),
        "fraction_at_0.01_pct": 100 * np.isclose(values[valid], .01, atol=.001).mean(),
        "fraction_below_0.05_pct": 100 * (values[valid] <= .05).mean(),
        "reference_median_of_observed_pixels": median(baseline_same_pixels),
    })
    grid = np.full(land.shape, np.nan)
    grid.ravel()[land_pixels[support_idx]] = values
    case_maps.append(grid)
reference_on_low_support = np.full(land.shape, np.nan)
low_valid = np.isfinite(case_raw[0, support_idx])
reference_on_low_support.ravel()[land_pixels[support_idx[low_valid]]] = (
    ntl0[support_idx[low_valid]]
)
display(pd.DataFrame(case_rows).round(3))

fig = make_subplots(rows=1, cols=3, horizontal_spacing=.035, subplot_titles=[
    f"Low day · {case_dates[0]:%d %b %Y}",
    "60-day reference · same visible pixels",
    f"Comparison day · {case_dates[1]:%d %b %Y}",
])
for col, grid in enumerate([case_maps[0], reference_on_low_support, case_maps[1]], 1):
    fig.add_trace(go.Heatmap(
        x=a2.x.values, y=a2.y.values, z=grid,
        colorscale="Inferno", zmin=0, zmax=NTL_DISPLAY_MAX,
        showscale=(col == 3),
        colorbar=dict(title="NTL", thickness=16, tickfont=dict(size=14)),
        hovertemplate="%{x:.3f}°E, %{y:.3f}°N<br>NTL %{z:.2f}<extra></extra>",
    ), row=1, col=col)
    fig.update_xaxes(title_text="Longitude", nticks=3, row=1, col=col)
    fig.update_yaxes(title_text="Latitude" if col == 1 else None,
                     nticks=4, scaleanchor="x" if col == 1 else f"x{col}",
                     scaleratio=1, row=1, col=col)
style_figure(fig, "", 1500, 590)
fig.update_annotations(font=dict(size=16, color=PLOT_TEXT_COLOR))
finish_figure(fig, "baseline_low_day_pixel_composition")

In [ ]:
regional_week = daily.loc[
    daily.profile.eq("Regional G3")
    & daily.date.between("2012-02-04", "2012-02-10")
].sort_values("date")

raw_week, _, _ = read_days(dates.get_indexer(regional_week.date))
support = fixed["Regional G3"]

rows = []
for j, day in enumerate(regional_week.itertuples()):
    pixels = raw_week[j, support]
    valid = np.isfinite(pixels)
    values = pixels[valid]

    rows.append({
        "date": day.date,
        "SC_pct": day.spatial_coverage_pct,
        "valid_pixels": valid.sum(),
        "daily_median": np.median(values) if len(values) else np.nan,
        "pixels_at_0.01_pct": (
            100 * np.isclose(values, 0.01, atol=1e-6).mean()
            if len(values) else np.nan
        ),
        "pixels_below_0.05_pct": (
            100 * (values <= 0.05).mean()
            if len(values) else np.nan
        ),
        "reference_median_of_observed_pixels": (
            np.median(ntl0[support][valid]) if valid.any() else np.nan
        ),
    })

display(pd.DataFrame(rows).round(2))

regional_4d = four_day.loc[
    four_day.profile.eq("Regional G3") & four_day.observed
]
print(
    "Four-day RQ medians ≤0.05:",
    regional_4d.raw_median_ntl.le(0.05).sum(),
    "of", len(regional_4d), "observed blocks"
)

In [ ]:
class_check = daily.loc[
    daily.date.between("2012-02-05", "2012-02-09")
    & daily.class_code.isin(["11", "12", "13", "21", "22", "23", "30"])
]

display(
    class_check.pivot(
        index="date", columns="class_code", values="raw_uncapped"
    ).round(3)
)
display(
    class_check.pivot(
        index="date", columns="class_code", values="spatial_coverage_pct"
    ).round(1)
)

## 5. Fixed dim and bright pixel controls

Choose controls **before examining their time series** using only the existing 60-day reference and GHSL: dim rural pixels (classes 11–13) and bright urban-centre pixels (class 30). These are candidate controls, not proven stable or truly unlit sites. Their fresh MQF = 0 DNB-BRDF values are extracted directly; the main regional P95 cap is deliberately not imposed on this independent pixel diagnostic. Four-day medians are formed per pixel, then across the fixed group. A stricter **50% group SC gate** prevents one visible control pixel from defining the group median. The original regional RQ series keeps its 10% gate.

If a similar cycle survives in dim pixels with good support, examine shared observation or processing influences; that finding alone does not establish a lunar cause. If only bright pixels cycle, investigate lighting behaviour and brightness-dependent retrieval effects.

In [ ]:
CONTROL_PIXELS = 12
CONTROL_SC_GATE = 50.0
dark_pool = np.flatnonzero(
    reference_ok & np.isin(ghsl_vector, [11, 12, 13]) & (ntl0 <= .20)
)
bright_pool = np.flatnonzero(
    reference_ok & (ghsl_vector == 30) & (ntl0 >= 2.0)
)
# Keep the class contrast if a numerical threshold yields fewer than 12 candidates.
if len(dark_pool) < CONTROL_PIXELS:
    rural = np.flatnonzero(reference_ok & np.isin(ghsl_vector, [11, 12, 13]))
    dark_pool = rural[np.argsort(ntl0[rural])[:max(CONTROL_PIXELS, len(dark_pool))]]
if len(bright_pool) < CONTROL_PIXELS:
    urban = np.flatnonzero(reference_ok & (ghsl_vector == 30))
    bright_pool = urban[np.argsort(ntl0[urban])[-max(CONTROL_PIXELS, len(bright_pool)):]]
if not len(dark_pool) or not len(bright_pool):
    raise ValueError("No eligible dim rural or bright urban-centre reference pixels.")
rng = np.random.default_rng(20261008)
dark_idx = np.sort(rng.choice(dark_pool, size=min(CONTROL_PIXELS, len(dark_pool)), replace=False))
bright_idx = np.sort(rng.choice(bright_pool, size=min(CONTROL_PIXELS, len(bright_pool)), replace=False))
control_idx = np.r_[dark_idx, bright_idx]
control_group = np.array(["Dim rural"] * len(dark_idx) + ["Bright urban"] * len(bright_idx))
flat = land_pixels[control_idx]
iy, ix = np.unravel_index(flat, land.shape)
controls = pd.DataFrame({
    "group": control_group, "land_index": control_idx,
    "ghsl": ghsl_vector[control_idx].astype(int),
    "reference_ntl": ntl0[control_idx],
    "longitude": a2.x.values[ix], "latitude": a2.y.values[iy],
})
display(controls.round(3))

baseline_indices = np.flatnonzero(dates.to_series().between(
    CHARACTERIZATION_START, PRE_EVENT_END
).to_numpy())
y_coords = xr.DataArray(iy, dims="control_pixel")
x_coords = xr.DataArray(ix, dims="control_pixel")
control_raw = a2[DNB_BAND].isel(
    date=baseline_indices, y=y_coords, x=x_coords
).transpose("date", "control_pixel").compute().values.astype(float)
control_mqf = a2[MQF_BAND].isel(
    date=baseline_indices, y=y_coords, x=x_coords
).transpose("date", "control_pixel").compute().values
control_raw[
    (control_mqf != 0) | ~np.isfinite(control_raw)
    | (control_raw < 0) | np.isin(control_raw, KNOWN_FILL_VALUES)
] = np.nan

control_dates = dates[baseline_indices]
control_blocks = day_blocks[baseline_indices]
control_rows = []
for block in np.unique(control_blocks):
    start = EVENT_DATE + pd.Timedelta(days=int(block) * AGGREGATION_DAYS)
    end = start + pd.Timedelta(days=AGGREGATION_DAYS - 1)
    if start < CHARACTERIZATION_START or end > PRE_EVENT_END:
        continue
    pixel_4d = median(control_raw[control_blocks == block], axis=0)
    for name in ("Dim rural", "Bright urban"):
        group_values = pixel_4d[control_group == name]
        n_valid = int(np.isfinite(group_values).sum())
        sc_pct = 100 * n_valid / len(group_values)
        control_rows.append({
            "date": start + pd.Timedelta(days=1.5), "group": name,
            "median_ntl": median(group_values) if sc_pct >= CONTROL_SC_GATE else np.nan,
            "sc_pct": sc_pct, "n_valid": n_valid,
        })
control_four = pd.DataFrame(control_rows)
control_audit = control_four.groupby("group").agg(
    observed_blocks=("median_ntl", "count"),
    median_sc_pct=("sc_pct", "median"),
)
control_audit["selected_pixels"] = controls.groupby("group").size()
display(control_audit.round(1))

In [ ]:
control_wide = control_four.pivot(index="date", columns="group", values="median_ntl")
control_sc = control_four.pivot(index="date", columns="group", values="sc_pct")
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    row_heights=[.12, .44, .44], vertical_spacing=.055,
    subplot_titles=["Control observability", "Bright urban controls", "Dim rural controls"])
fig.add_trace(go.Heatmap(
    x=control_sc.index, y=["Bright urban", "Dim rural"],
    z=control_sc[["Bright urban", "Dim rural"]].T.to_numpy(),
    colorscale=SC_GREEN_COLORSCALE, zmin=0, zmax=100,
    colorbar=dict(title="SC (%)", thickness=16, tickfont=dict(size=14), len=.19, y=.9),
    hovertemplate="%{y}<br>%{x|%d %b %Y}<br>SC %{z:.0f}%<extra></extra>",
), row=1, col=1)
for row, name, color in [(2, "Bright urban", "#D55E00"),
                         (3, "Dim rural", "#7B5EA7")]:
    series = control_wide[name]
    fig.add_trace(go.Scatter(
        x=series.index, y=series, mode="lines", connectgaps=False,
        line=dict(color=color, width=2.0, shape="hv"), name=name,
        hovertemplate=f"{name}<br>%{{x|%d %b %Y}}<br>NTL %{{y:.3f}}<extra></extra>",
    ), row=row, col=1)
    fig.update_yaxes(title_text="NTL (nW cm⁻² sr⁻¹)", rangemode="tozero", row=row, col=1)
style_figure(fig, "", 1500, 770)
for row in (1, 2, 3):
    fig.update_xaxes(range=[CHARACTERIZATION_START, PRE_EVENT_END],
        tickmode="array", tickvals=quarter_ticks,
        ticktext=[d.strftime("%b %Y") for d in quarter_ticks],
        showticklabels=row == 3, row=row, col=1)
    for month in month_starts:
        fig.add_vline(x=month, line_color="rgba(36,59,90,.10)",
                      line_width=.8, layer="below", row=row, col=1)
fig.update_annotations(font=dict(size=17, color=PLOT_TEXT_COLOR))
finish_figure(fig, "baseline_dim_bright_controls")

paired = control_wide.dropna()
print(f"Paired qualified blocks: {len(paired)}")
print("Bright–dim correlation on paired blocks:",
      round(paired["Bright urban"].corr(paired["Dim rural"]), 3) if len(paired) > 2 else "insufficient pairs")

## 6. Locate candidate frequencies before assigning an STL period

Lomb–Scargle uses only observed four-day values, so it does not create a wave across missing blocks. The ordinary Fourier periodogram below is deliberately applied to the **interpolated** regional input as a sensitivity comparison, not as independent confirmation. The lower panel shows the observation-window and SC spectra. A peak near 28 days may reflect the four-day aggregation/STL choice; a peak near 29.53 days is only a lunar-period *candidate*. Shared peaks with observability require particular caution. There are fewer than two pre-Haiyan annual cycles, so annual power is poorly resolved.

In [ ]:
from scipy.signal import lombscargle, periodogram

period_days = np.linspace(8, 180, 1400)
omega = 2 * np.pi / period_days
series_for_spectrum = {
    "Regional G3 · RQ (4D)": pre.loc[pre.profile.eq("Regional G3"),
        ["date", "raw_median_ntl"]].rename(columns={"raw_median_ntl": "value"}),
    "Bright urban controls": control_wide["Bright urban"].rename("value").reset_index(),
    "Dim rural controls": control_wide["Dim rural"].rename("value").reset_index(),
}
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    row_heights=[.68, .32], vertical_spacing=.10,
    subplot_titles=["Observed-value periodicity", "Observability periodicity"])
peak_rows = []
for name, frame in series_for_spectrum.items():
    g = frame.dropna(subset=["value"]).sort_values("date")
    if len(g) < 40:
        continue
    time = (g.date - CHARACTERIZATION_START).dt.total_seconds().to_numpy() / 86400
    values = np.log1p(g.value.to_numpy(dtype=float))
    values = values - values.mean()
    if np.var(values) <= 1e-12:
        continue
    power = lombscargle(time, values, omega, normalize=True)
    color = {"Regional G3 · RQ (4D)": RQ_FOUR_DAY_COLOR,
             "Bright urban controls": "#D55E00", "Dim rural controls": "#7B5EA7"}[name]
    fig.add_trace(go.Scatter(x=period_days, y=power, mode="lines",
        line=dict(color=color, width=2), name=name,
        hovertemplate=f"{name}<br>Period %{{x:.1f}} d<br>Power %{{y:.3f}}<extra></extra>"),
        row=1, col=1)
    peak_rows.append({"series": name, "observed_blocks": len(g),
        "peak_period_days": period_days[np.argmax(power)],
        "peak_power": np.nanmax(power)})

regional_pre = pre.loc[pre.profile.eq("Regional G3")].sort_values("date")
regular = regional_pre.set_index("date").raw_median_ntl
filled = regular.interpolate(method="time", limit_area="inside").ffill().bfill()
if filled.notna().all() and len(filled) > 40:
    f, fft_power = periodogram(np.log1p(filled.to_numpy()), fs=1/4, detrend="linear")
    fft_period = np.divide(1, f, out=np.full_like(f, np.inf), where=f > 0)
    select = (f > 0) & (fft_period >= 8) & (fft_period <= 180)
    if np.any(select) and fft_power[select].max() > 0:
        fig.add_trace(go.Scatter(x=fft_period[select],
            y=fft_power[select]/fft_power[select].max(), mode="lines",
            line=dict(color="#6C7882", width=1.4, dash="dot"),
            name="Interpolated-input FFT · relative power"), row=1, col=1)

calendar_time = (regional_pre.date - CHARACTERIZATION_START).dt.total_seconds().to_numpy()/86400
for label, values, color in [
    ("Observed-block indicator", regional_pre.raw_median_ntl.notna().astype(float).to_numpy(), "#243B5A"),
    ("Spatial completeness", regional_pre.spatial_coverage_pct.fillna(0).to_numpy()/100, "#009E73"),
]:
    centered = values - values.mean()
    if np.var(centered) > 0:
        power = lombscargle(calendar_time, centered, omega, normalize=True)
        fig.add_trace(go.Scatter(x=period_days, y=power, mode="lines",
            line=dict(color=color, width=1.8), name=label), row=2, col=1)
for row in (1, 2):
    for candidate, color, label in [(28, "#E69F00", "28 d"),
                                    (29.53, "#B33D6A", "29.53 d")]:
        fig.add_vline(x=candidate, line_color=color, line_dash="dash",
                      line_width=1.2, row=row, col=1)
fig.update_xaxes(title_text="Candidate period (days)", range=[8, 180], row=2, col=1)
fig.update_yaxes(title_text="Normalized power", row=1, col=1)
fig.update_yaxes(title_text="Normalized power", row=2, col=1)
style_figure(fig, "", 1500, 760)
fig.update_annotations(font=dict(size=17, color=PLOT_TEXT_COLOR))
finish_figure(fig, "baseline_observed_and_observability_spectra")
display(pd.DataFrame(peak_rows).round(2))

## 7. STL is a description of a specified period, not proof of its cause

The retained regional and GHSL-class plots use the original four-day calendar and a 7-block (28-day) STL period. Missing qualified blocks are interpolated **only for the fit**; solid traces mark observed support and dotted spans mark supplied input. Trend and seasonal/residual components have different units: trend is back-transformed to NTL, while seasonal and residual remain in log1p units. A slow annual-looking pattern may be absorbed into the STL trend. The next figure repeats STL on daily input for physically interpretable assumed periods of 7, 14, 28, 60, 90 and 120 days; different curves indicate model sensitivity, not six detected physical cycles.

In [ ]:
# Regional Samar–Leyte G3: reliability-qualified NTL and exploratory STL
from statsmodels.tsa.seasonal import STL

regional_daily = pd.read_csv(
    cache_dir / "daily.csv",
    usecols=[
        "date", "profile", "raw_uncapped",
        "valid_pixels", "spatial_coverage_pct",
    ],
    parse_dates=["date"],
)
regional_daily = regional_daily.loc[
    regional_daily["profile"].eq("Regional G3")
    & regional_daily["date"].between(
        CHARACTERIZATION_START, CHARACTERIZATION_END
    )
].sort_values("date")

regional_blocks = pd.read_csv(
    cache_dir / "four_day.csv",
    usecols=[
        "date", "profile", "rq_median",
        "valid_pixels", "spatial_coverage_pct",
    ],
    parse_dates=["date"],
)
regional_blocks = regional_blocks.loc[
    regional_blocks["profile"].eq("Regional G3")
].copy()

# Include only four-day blocks wholly inside the pre-Haiyan period.
regional_blocks["date_start"] = (
    regional_blocks["date"] - pd.Timedelta(days=1.5)
)
regional_blocks["date_end"] = (
    regional_blocks["date"] + pd.Timedelta(days=1.5)
)
regional_blocks = regional_blocks.loc[
    regional_blocks["date_start"].ge(CHARACTERIZATION_START)
    & regional_blocks["date_end"].le(CHARACTERIZATION_END)
].sort_values("date")

calendar = pd.DatetimeIndex(regional_blocks["date"])
sc = pd.Series(
    regional_blocks["spatial_coverage_pct"].to_numpy(),
    index=calendar,
)
qualified = (
    regional_blocks["spatial_coverage_pct"].ge(
        SPATIAL_COMPLETENESS_PCT
    )
    & regional_blocks["valid_pixels"].gt(0)
)
rq = pd.Series(
    regional_blocks["rq_median"].where(qualified).to_numpy(),
    index=calendar,
)

if rq.notna().sum() < 14:
    raise ValueError("Too few observed regional G3 blocks for 28-day STL.")

# Missing blocks receive values solely to make a regular STL input.
# The observed RQ series remains unchanged.
stl_input = (
    rq.interpolate(method="time", limit_area="inside")
    .ffill()
    .bfill()
)
observed = rq.notna()
missing = ~observed
gap_groups = missing.ne(missing.shift(fill_value=False)).cumsum()
longest_gap = int(missing.groupby(gap_groups).sum().max())

display(pd.DataFrame([{
    "profile": "Regional G3",
    "calendar_blocks": len(calendar),
    "observed_RQ_blocks": int(observed.sum()),
    "blocks_supplied_to_STL": int(missing.sum()),
    "supplied_pct": round(100 * missing.mean(), 1),
    "longest_gap_blocks": longest_gap,
    "SC_threshold_pct": SPATIAL_COMPLETENESS_PCT,
}]))

# Seven four-day blocks represent a 28-day diagnostic component.
stl = STL(np.log1p(stl_input.to_numpy()), period=7, robust=True).fit()
trend = pd.Series(np.expm1(stl.trend), index=calendar)
seasonal = pd.Series(stl.seasonal, index=calendar)
residual = pd.Series(stl.resid, index=calendar)

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    row_heights=[.25, .375, .225, .15],
    vertical_spacing=.055,
    subplot_titles=[
        "Observed nighttime lights",
        "Trend",
        "Seasonality · 28-day diagnostic",
        "Residual",
    ],
)

fig.add_trace(go.Scatter(
    x=regional_daily["date"],
    y=regional_daily["raw_uncapped"],
    mode="markers",
    marker=dict(color=DNB_COLOR, size=4, opacity=.38),
    name="Daily DNB-BRDF NTL",
    customdata=regional_daily[
        ["valid_pixels", "spatial_coverage_pct"]
    ].to_numpy(),
    hovertemplate=(
        "%{x|%d %b %Y}<br>Daily DNB-BRDF NTL: %{y:.2f}"
        "<br>Valid pixels: %{customdata[0]:.0f}"
        "<br>SC: %{customdata[1]:.0f}%<extra></extra>"
    ),
), row=1, col=1)

# The solid observed series covers this dashed input wherever RQ data exist.
# Visible dashed spans therefore identify blocks supplied to STL.
fig.add_trace(go.Scatter(
    x=calendar,
    y=stl_input,
    mode="lines",
    line=dict(color=BASELINE_COLOR, width=1.6, dash="dot"),
    name="STL input",
    hovertemplate="%{x|%d %b %Y}<br>STL input: %{y:.2f}<extra></extra>",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=calendar,
    y=rq,
    mode="lines+markers",
    connectgaps=False,
    line=dict(color=RQ_FOUR_DAY_COLOR, width=2.6),
    marker=dict(color=RQ_FOUR_DAY_COLOR, size=4),
    name="RQ NTL (4D)",
    customdata=sc.to_numpy().reshape(-1, 1),
    hovertemplate=(
        "%{x|%d %b %Y}<br>RQ NTL (4D): %{y:.2f}"
        "<br>SC: %{customdata[0]:.0f}%<extra></extra>"
    ),
), row=1, col=1)

for row, values, color, label in [
    (2, trend, "#174A7E", "Trend"),
    (3, seasonal, "#E69F00", "Seasonality"),
    (4, residual, "#6C7882", "Residual"),
]:
    # Dashed portions show the decomposition through supplied input blocks.
    fig.add_trace(go.Scatter(
        x=calendar,
        y=values,
        mode="lines",
        line=dict(color=color, width=1.7, dash="dot"),
        opacity=.7,
        showlegend=False,
        hovertemplate=(
            f"%{{x|%d %b %Y}}<br>{label}: %{{y:.3f}}"
            "<br>STL model input<extra></extra>"
        ),
    ), row=row, col=1)

    fig.add_trace(go.Scatter(
        x=calendar,
        y=values.where(observed),
        mode="lines",
        connectgaps=False,
        line=dict(color=color, width=2.7),
        showlegend=False,
        hovertemplate=(
            f"%{{x|%d %b %Y}}<br>{label}: %{{y:.3f}}"
            "<br>Observed RQ block<extra></extra>"
        ),
    ), row=row, col=1)

for row in (3, 4):
    fig.add_hline(
        y=0, line_color="#AAB5C2", line_width=1,
        row=row, col=1,
    )

fig.update_yaxes(
    title_text="NTL (nW cm⁻² sr⁻¹)", rangemode="tozero",
    row=1, col=1,
)
fig.update_yaxes(
    title_text="Trend NTL<br>(nW cm⁻² sr⁻¹)", rangemode="tozero",
    row=2, col=1,
)
fig.update_yaxes(
    title_text="Seasonality<br>(log1p NTL)",
    row=3, col=1,
)
fig.update_yaxes(
    title_text="Residual<br>(log1p NTL)",
    row=4, col=1,
)

style_figure(fig, "", 1600, 980)

for row in range(1, 5):
    fig.update_xaxes(
        range=[display_start, CHARACTERIZATION_END],
        showgrid=False,
        tickmode="array",
        tickvals=quarter_ticks,
        ticktext=[d.strftime("%b %Y") for d in quarter_ticks],
        tickfont=dict(size=14),
        showticklabels=(row == 4),
        row=row, col=1,
    )
    fig.update_yaxes(tickfont=dict(size=14), row=row, col=1)

    for month in month_starts:
        fig.add_vline(
            x=month,
            line_color=(
                "rgba(36,59,90,.20)"
                if month.month == 1
                else "rgba(36,59,90,.10)"
            ),
            line_width=1.2 if month.month == 1 else .8,
            layer="below",
            row=row, col=1,
        )

fig.update_layout(
    legend=dict(
        orientation="h", x=0, y=-.08,
        xanchor="left", yanchor="top",
        font=dict(size=16),
    ),
)
fig.update_annotations(font=dict(size=17, color=PLOT_TEXT_COLOR))
finish_figure(fig, "regional_g3_observed_and_stl")

In [ ]:
# Exploratory GHSL-class STL using the full pre-Haiyan four-day calendar
# Missing blocks are interpolated for this diagnostic, never set to zero.

from statsmodels.tsa.seasonal import STL

STL_PERIOD_BLOCKS = 7  # 28-day component; not annual seasonality
class_groups = {
    1: (11, 12, 13, 21),
    2: (22, 23, 30),
}

calendar = pd.DatetimeIndex(sorted(pre.date.dropna().unique()))
stl_frames = []
audit_rows = []

for code in (11, 12, 13, 21, 22, 23, 30):
    name = f"{code} · {GHSL_CLASS_LABELS[code]}"
    g = pre[pre.profile.eq(name)].sort_values("date")

    observed = (
        g.set_index("date").raw_median_ntl
        .reindex(calendar)
    )

    if observed.notna().sum() < 2:
        audit_rows.append(dict(
            ghsl=code, observed_blocks=int(observed.notna().sum()),
            interpolated_blocks=np.nan, status="Too few observations",
        ))
        continue

    missing = observed.isna()
    gap_id = missing.ne(missing.shift(fill_value=False)).cumsum()
    longest_gap = max(
        (len(run) for _, run in missing.groupby(gap_id) if run.all()),
        default=0,
    )

    # Interior gaps: linear interpolation in calendar time.
    # Edge gaps: carry the nearest observed level to complete the STL input.
    filled = (
        observed.interpolate(method="time", limit_area="inside")
        .ffill()
        .bfill()
    )

    result = STL(
        np.log1p(filled),
        period=STL_PERIOD_BLOCKS,
        robust=True,
    ).fit()

    stl_frames.append(pd.DataFrame({
        "date": calendar,
        "ghsl": code,
        "observed_input": ~missing.to_numpy(),
        "trend": np.asarray(result.trend),
        "seasonal": np.asarray(result.seasonal),
        "residual": np.asarray(result.resid),
    }))

    audit_rows.append(dict(
        ghsl=code,
        observed_blocks=int((~missing).sum()),
        interpolated_blocks=int(missing.sum()),
        interpolated_pct=100 * missing.mean(),
        longest_gap_blocks=longest_gap,
        status="STL fitted",
    ))

stl_audit = pd.DataFrame(audit_rows)
display(stl_audit.round(1))

if not stl_frames:
    print("No GHSL class has enough observed blocks for STL.")
else:
    stl_classes = pd.concat(stl_frames, ignore_index=True)

    fig = make_subplots(
        rows=3, cols=2,
        shared_xaxes=True,
        row_heights=[.50, .30, .20],
        horizontal_spacing=.10,
        vertical_spacing=.07,
        subplot_titles=[
            "Classes 11–21 · trend", "Classes 22–30 · trend",
            "Classes 11–21 · 28-day component",
            "Classes 22–30 · 28-day component",
            "Classes 11–21 · residual",
            "Classes 22–30 · residual",
        ],
    )

    for col, codes in class_groups.items():
        for code in codes:
            g = stl_classes[
                stl_classes.ghsl.eq(code)
            ].sort_values("date")
            if g.empty:
                continue

            color = GHSL_CLASS_COLORS[code]
            for row, component in enumerate(
                ("trend", "seasonal", "residual"), start=1
            ):
                # Dotted line completes the component through filled blocks.
                fig.add_trace(
                    go.Scatter(
                        x=g.date,
                        y=g[component],
                        mode="lines",
                        line=dict(color=color, width=1.8, dash="dot"),
                        opacity=.65,
                        showlegend=False,
                        hovertemplate=(
                            f"GHSL {code}<br>%{{x|%d %b %Y}}"
                            f"<br>{component.capitalize()} %{{y:.3f}}"
                            "<br>STL includes interpolated input"
                            "<extra></extra>"
                        ),
                    ),
                    row=row, col=col,
                )

                # Solid line appears only at originally observed blocks.
                observed_component = g[component].where(g.observed_input)
                fig.add_trace(
                    go.Scatter(
                        x=g.date,
                        y=observed_component,
                        mode="lines",
                        line=dict(color=color, width=2.8),
                        connectgaps=False,
                        name=f"{code} · {GHSL_CLASS_LABELS[code]}",
                        legendgroup=str(code),
                        showlegend=(row == 1),
                        hovertemplate=(
                            f"GHSL {code}<br>%{{x|%d %b %Y}}"
                            f"<br>{component.capitalize()} %{{y:.3f}}"
                            "<br>Originally observed RQ block"
                            "<extra></extra>"
                        ),
                    ),
                    row=row, col=col,
                )

        for row in (2, 3):
            fig.add_hline(
                y=0, line_color=PLOT_GRID_COLOR,
                line_width=1, row=row, col=col,
            )

    for row, label in [
        (1, "Trend · log1p NTL"),
        (2, "28-day component · log1p"),
        (3, "Residual · log1p"),
    ]:
        fig.update_yaxes(
            title_text=label, tickfont=dict(size=14),
            row=row, col=1,
        )
        fig.update_yaxes(
            tickfont=dict(size=14),
            row=row, col=2,
        )

    for col in (1, 2):
        fig.update_xaxes(
            range=[calendar.min(), calendar.max()],
            tickformat="%b %Y",
            dtick="M3",
            tickfont=dict(size=14),
            row=3, col=col,
        )

    # Legend key for the dotted portions.
    fig.add_trace(
        go.Scatter(
            x=[None], y=[None], mode="lines",
            line=dict(color=BASELINE_COLOR, dash="dot", width=2),
            name="Interpolated-input portion",
            showlegend=True,
        ),
        row=1, col=1,
    )

    style_figure(fig, "", 1850, 1280)
    fig.update_layout(
        margin=dict(l=105, r=45, t=85, b=170),
        legend=dict(
            orientation="h",
            x=0, y=-.10,
            xanchor="left", yanchor="top",
            font=dict(size=16),
        ),
    )
    fig.update_annotations(font=dict(size=17, color=PLOT_TEXT_COLOR))
    finish_figure(fig, "ghsl_four_day_stl_full_calendar")

In [ ]:
daily_regional = daily.loc[
    daily.profile.eq("Regional G3")
    & daily.date.between(CHARACTERIZATION_START, PRE_EVENT_END)
].set_index("date").sort_index()
daily_calendar = pd.date_range(CHARACTERIZATION_START, PRE_EVENT_END, freq="D")
daily_observed = daily_regional.raw_median_ntl.reindex(daily_calendar)
daily_sc = daily_regional.spatial_coverage_pct.reindex(daily_calendar)
daily_input = daily_observed.interpolate(method="time", limit_area="inside").ffill().bfill()
if daily_observed.notna().sum() < 120:
    raise ValueError("Too few qualified daily G3 observations for this period screen.")
daily_periods = {7: "#0072B2", 14: "#009E73", 28: "#E69F00",
                 60: "#D55E00", 90: "#CC79A7", 120: "#243B5A"}
daily_stl_rows = []
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    row_heights=[.13, .87], vertical_spacing=.06,
    subplot_titles=["Regional observability", "STL trend by assumed daily period"])
fig.add_trace(go.Heatmap(x=daily_calendar, y=["SC"], z=[daily_sc.fillna(0).to_numpy()],
    colorscale=SC_GREEN_COLORSCALE, zmin=0, zmax=100,
    colorbar=dict(title="SC (%)", thickness=16, tickfont=dict(size=14), len=.18, y=.91)),
    row=1, col=1)
fig.add_trace(go.Scatter(x=daily_calendar, y=daily_observed,
    mode="markers", marker=dict(color="#929BA5", size=3, opacity=.30),
    name="RQ NTL · daily"), row=2, col=1)
for period_days_setting, color in daily_periods.items():
    result = STL(np.log1p(daily_input.to_numpy()),
                 period=period_days_setting, robust=True).fit()
    trend_ntl = np.expm1(result.trend)
    observed_mask = daily_observed.notna().to_numpy()
    remainder = np.asarray(result.resid)[observed_mask]
    seasonal_component = np.asarray(result.seasonal)[observed_mask]
    total = remainder + seasonal_component
    strength = max(0, 1 - np.var(remainder)/np.var(total)) if np.var(total) > 0 else np.nan
    daily_stl_rows.append({"assumed_period_days": period_days_setting,
        "seasonal_strength_observed": strength,
        "observed_days": int(observed_mask.sum()),
        "interpolated_pct": 100 * (1 - observed_mask.mean())})
    fig.add_trace(go.Scatter(x=daily_calendar, y=trend_ntl, mode="lines",
        line=dict(color=color, width=1.2 + period_days_setting/65),
        name=f"{period_days_setting} d"), row=2, col=1)
style_figure(fig, "", 1500, 720)
fig.update_yaxes(title_text="NTL (nW cm⁻² sr⁻¹)", rangemode="tozero", row=2, col=1)
fig.update_xaxes(range=[CHARACTERIZATION_START, PRE_EVENT_END],
    tickmode="array", tickvals=quarter_ticks,
    ticktext=[d.strftime("%b %Y") for d in quarter_ticks], row=2, col=1)
for month in month_starts:
    fig.add_vline(x=month, line_color="rgba(36,59,90,.10)",
                  line_width=.8, layer="below", row=2, col=1)
fig.update_annotations(font=dict(size=17, color=PLOT_TEXT_COLOR))
finish_figure(fig, "baseline_daily_stl_period_sensitivity")
display(pd.DataFrame(daily_stl_rows).round(3))

## 8. Breakpoint screen and optional BFAST

An abrupt pre-event change can masquerade as smooth seasonality. The first cell is a **BFAST-inspired**, observation-only screen: compare a 28-day harmonic plus linear trend with the same model allowing one level and slope break. It uses BIC for a descriptive ranking; serial correlation and the retrospective search prevent a formal significance claim. The next cell optionally runs the actual R `bfast` package on the regularized 28-day input if R and the package are already installed. It is skipped otherwise, without installing software. BFAST on interpolated input is a sensitivity check, not a replacement for the observation-only screen.

In [ ]:
g = pre.loc[pre.profile.eq("Regional G3") & pre.raw_median_ntl.notna()].sort_values("date")
t = (g.date - CHARACTERIZATION_START).dt.total_seconds().to_numpy()/86400
y = np.log1p(g.raw_median_ntl.to_numpy(dtype=float))
harmonic = np.column_stack([np.ones(len(t)), t,
    np.sin(2*np.pi*t/28), np.cos(2*np.pi*t/28)])
def fit_bic(design, response):
    beta = np.linalg.lstsq(design, response, rcond=None)[0]
    fitted = design @ beta
    rss = max(np.sum((response-fitted)**2), 1e-12)
    return len(response)*np.log(rss/len(response)) + design.shape[1]*np.log(len(response)), fitted
base_bic, base_fit = fit_bic(harmonic, y)
candidate_days = np.arange(120, (PRE_EVENT_END-CHARACTERIZATION_START).days-120, 4)
screen = []
for break_day in candidate_days:
    post = (t >= break_day).astype(float)
    if post.sum() < 25 or (1-post).sum() < 25:
        continue
    design = np.column_stack([harmonic, post, np.maximum(0, t-break_day)])
    bic, fitted = fit_bic(design, y)
    screen.append((bic, break_day, fitted))
best_bic, best_break, best_fit = min(screen, key=lambda row: row[0])
break_date = CHARACTERIZATION_START + pd.Timedelta(days=int(best_break))
display(pd.DataFrame([{
    "observed_blocks": len(g), "no_break_BIC": base_bic,
    "best_one_break_BIC": best_bic, "BIC_improvement": base_bic-best_bic,
    "candidate_break_date": break_date,
    "interpretation": "screen only; inspect SC and event records",
}]).round(2))

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    row_heights=[.13, .87], vertical_spacing=.06)
g_sc = pre.loc[pre.profile.eq("Regional G3")].sort_values("date")
fig.add_trace(go.Heatmap(x=g_sc.date, y=["SC"], z=[g_sc.spatial_coverage_pct],
    colorscale=SC_GREEN_COLORSCALE, zmin=0, zmax=100,
    colorbar=dict(title="SC (%)", thickness=16, tickfont=dict(size=14), len=.18, y=.91)),
    row=1, col=1)
for values, name, color, width in [
    (g.raw_median_ntl, "Observed RQ NTL (4D)", RQ_FOUR_DAY_COLOR, 1.2),
    (np.expm1(base_fit), "No-break harmonic", "#243B5A", 2.3),
    (np.expm1(best_fit), "Best one-break harmonic", "#D55E00", 2.3),
]:
    fig.add_trace(go.Scatter(x=g.date, y=values, mode="lines",
        connectgaps=False, line=dict(color=color, width=width), name=name), row=2, col=1)
fig.add_vline(x=break_date, line_color="#B33D6A", line_dash="dash",
              line_width=1.5, row=2, col=1)
style_figure(fig, "", 1500, 660)
fig.update_yaxes(title_text="NTL (nW cm⁻² sr⁻¹)", rangemode="tozero", row=2, col=1)
fig.update_xaxes(range=[CHARACTERIZATION_START, PRE_EVENT_END], row=2, col=1)
finish_figure(fig, "baseline_one_break_harmonic_screen")

In [ ]:
import shutil
import subprocess

if shutil.which("Rscript") is None:
    print("Actual BFAST skipped: Rscript is not installed. The Python breakpoint screen above still runs.")
else:
    package_check = subprocess.run(
        ["Rscript", "--vanilla", "-e", "cat(requireNamespace('bfast', quietly=TRUE))"],
        capture_output=True, text=True, timeout=30,
    )
    if package_check.returncode or package_check.stdout.strip() != "TRUE":
        print("Actual BFAST skipped: the R 'bfast' package is unavailable.")
    else:
        bfast_input = TABLE_DIR / "baseline_bfast_28d_regularized_input.csv"
        pd.DataFrame({"date": regular.index,
                      "ntl_log": np.log1p(filled.to_numpy())}).to_csv(bfast_input, index=False)
        r_script = r"""
args <- commandArgs(trailingOnly=TRUE)
d <- read.csv(args[1])
stopifnot(all(is.finite(d$ntl_log)))
y <- ts(d$ntl_log, frequency=7)  # seven four-day blocks = 28 days
fit <- bfast::bfast(y, h=0.15, season="harmonic", max.iter=1, breaks=1)
print(fit)
print(summary(fit))
"""
        result = subprocess.run(["Rscript", "--vanilla", "-e", r_script,
                                 str(bfast_input)], capture_output=True,
                                text=True, timeout=120)
        print(result.stdout if result.returncode == 0 else result.stderr[-3000:])

## 9. Named Tacloban pixels as the final baseline check

The retained paired maps show each selected 1×1 pixel against GHSL and pre-Haiyan NTL. The final time-series plot uses the original colours and site labels. Look for synchrony across different urban functions and lower-density comparison pixels, but keep the varying SC and the 60-day reference brightness in mind. These are identifiable locations, not independent ground measurements of electricity or lighting use.

In [ ]:
# 1. Select distinct, observable pixels and show their spatial context

site_specs = [
    ("Airport", 125.02645, 11.22747, "#0072B2", "diamond"),
    ("Port", 124.99865, 11.24905, "#E69F00", "square"),
    ("Main-road vicinity", 125.00300, 11.24200, "#009E73", "cross"),
    ("BLISS Sagkahan residential", 125.00264, 11.21767, "#CC79A7", "triangle-up"),
    ("Kassel City residential", 124.98314, 11.23742, "#7B5EA7", "triangle-down"),
    ("Rizal Plaza", 125.00614, 11.24208, "#D55E00", "star"),
    ("Astrodome", 125.00502, 11.22165, "#56B4E9", "x"),
    ("Robinsons Place", 125.00778, 11.20752, "#B33D6A", "hexagon"),
]

In [ ]:
# Select distinct, reference-qualified pixels for the named sites
# and nearby GHSL comparison pixels.

map_bounds = (124.91, 11.13, 125.10, 11.33)

eligible = np.flatnonzero(reference_ok)
flat = land_pixels[eligible]
iy, ix = np.unravel_index(flat, land.shape)

candidate = pd.DataFrame({
    "ix": ix,
    "iy": iy,
    "x": a2.x.values[ix],
    "y": a2.y.values[iy],
    "ghsl": ghsl_vector[eligible].astype(int),
    "reference_ntl": ntl0[eligible],
})
candidate = candidate.loc[
    candidate.x.between(map_bounds[0], map_bounds[2])
    & candidate.y.between(map_bounds[1], map_bounds[3])
].copy()

if candidate.empty:
    raise ValueError("No reference-qualified land pixels in the POI map bounds.")

selected_sites = []
used_pixels = set()

for name, lon, lat, color, symbol in site_specs:
    available = candidate.loc[~candidate.index.isin(used_pixels)]
    distance = 111.2 * np.hypot(
        (available.x - lon) * np.cos(np.deg2rad(lat)),
        available.y - lat,
    )
    chosen_id = distance.idxmin()
    shift_km = float(distance.loc[chosen_id])

    if shift_km > 1.5:
        raise ValueError(
            f"{name}: nearest eligible pixel is {shift_km:.2f} km away."
        )

    p = available.loc[chosen_id]
    used_pixels.add(chosen_id)
    selected_sites.append(dict(
        site=name,
        x=float(p.x), y=float(p.y),
        ix=int(p.ix), iy=int(p.iy),
        ghsl=int(p.ghsl),
        reference_ntl=float(p.reference_ntl),
        shift_km=shift_km,
        color=color, symbol=symbol,
    ))

comparison_symbols = {
    23: "diamond", 22: "square", 21: "triangle-up",
    13: "circle", 12: "triangle-down", 11: "star",
}

for code, symbol in comparison_symbols.items():
    available = candidate.loc[
        candidate.ghsl.eq(code)
        & ~candidate.index.isin(used_pixels)
    ]
    if available.empty:
        print(f"No eligible GHSL {code} pixel within the map bounds.")
        continue

    distance = 111.2 * np.hypot(
        (available.x - 125.0015) * np.cos(np.deg2rad(11.2434)),
        available.y - 11.2434,
    )
    chosen_id = distance.idxmin()
    p = available.loc[chosen_id]
    used_pixels.add(chosen_id)

    selected_sites.append(dict(
        site=f"Pixel {code}",
        x=float(p.x), y=float(p.y),
        ix=int(p.ix), iy=int(p.iy),
        ghsl=int(p.ghsl),
        reference_ntl=float(p.reference_ntl),
        shift_km=np.nan,
        color=GHSL_CLASS_COLORS[code],
        symbol=symbol,
    ))

site_df = pd.DataFrame(selected_sites)

# The later 4×4 POI figures use the eight named sites only.
poi_sites = site_df.loc[
    site_df.site.isin([spec[0] for spec in site_specs])
].copy()
poi_colors = poi_sites.color.tolist()
poi_symbols = poi_sites.symbol.tolist()

# Fixed reference-qualified support for the spatial-window checks.
poi_fixed = {}
for site in site_df.itertuples():
    for size in (1, 3, 5):
        radius = size // 2
        window = np.zeros(land.shape, dtype=bool)
        window[
            max(0, site.iy-radius):site.iy+radius+1,
            max(0, site.ix-radius):site.ix+radius+1,
        ] = True
        poi_fixed[(site.site, size)] = np.flatnonzero(
            window.ravel()[land_pixels] & reference_ok
        )

display(
    site_df[
        ["site", "ghsl", "reference_ntl", "shift_km", "x", "y"]
    ].round(3)
)


In [ ]:
# Selected 1×1 pixels on GHSL and pre-Haiyan reference NTL maps.
# Requires site_df and the display grids from the earlier map cell.

map_bounds = (124.91, 11.13, 125.10, 11.33)
xi = np.flatnonzero(
    (a2.x.values >= map_bounds[0]) & (a2.x.values <= map_bounds[2])
)
yi = np.flatnonzero(
    (a2.y.values >= map_bounds[1]) & (a2.y.values <= map_bounds[3])
)
dx = float(abs(a2.x.values[1] - a2.x.values[0]))
dy = float(abs(a2.y.values[1] - a2.y.values[0]))

fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=.14,
    subplot_titles=[
        "Settlement classes (GHSL)",
        "Pre-Haiyan nighttime lights (60-day median)",
    ])

for col, grid in ((1, ghsl_grid), (2, reference_grid)):
    # Show water on the NTL panel without treating missing land NTL as water.
    if col == 2:
        fig.add_trace(go.Heatmap(
            x=a2.x.values[xi],
            y=a2.y.values[yi],
            z=water_grid[np.ix_(yi, xi)],
            colorscale=[[0, "#B8D9EF"], [1, "#B8D9EF"]],
            showscale=False,
            hoverinfo="skip",
        ), row=1, col=col)

    fig.add_trace(go.Heatmap(
        x=a2.x.values[xi],
        y=a2.y.values[yi],
        z=grid[np.ix_(yi, xi)],
        colorscale=ghsl_scale if col == 1 else "Inferno",
        zmin=-.5 if col == 1 else NTL_DISPLAY_MIN,
        zmax=len(map_classes) - .5 if col == 1 else NTL_DISPLAY_MAX,
        zsmooth=False,
        hoverongaps=False,
        colorbar=dict(
            x=.445 if col == 1 else 1.01,
            len=.72,
            thickness=17,
            title=dict(
                text="GHSL" if col == 1 else "NTL<br>(nW cm⁻² sr⁻¹)",
                font=dict(size=16),
            ),
            tickfont=dict(size=14),
            tickvals=list(range(len(map_classes))) if col == 1 else None,
            ticktext=[str(code) for code, _, _ in map_classes]
                     if col == 1 else None,
        ),
        hovertemplate=(
            "GHSL %{z:.0f}<extra></extra>" if col == 1
            else "%{y:.4f}°N, %{x:.4f}°E"
                 "<br>NTL %{z:.2f}<extra></extra>"
        ),
    ), row=1, col=col)

    fig.add_trace(go.Scatter(
        x=outline_x, y=outline_y,
        mode="lines",
        line=dict(color="rgba(36,59,90,.45)", width=.7),
        hoverinfo="skip",
        showlegend=False,
    ), row=1, col=col)

    fig.update_xaxes(
        range=[map_bounds[0], map_bounds[2]],
        tickformat=".2f", ticksuffix="°E",
        showgrid=False, row=1, col=col,
    )
    fig.update_yaxes(
        range=[map_bounds[1], map_bounds[3]],
        tickformat=".2f", ticksuffix="°N",
        showgrid=False,
        scaleanchor="x" if col == 1 else "x2",
        scaleratio=1 / np.cos(np.deg2rad(11.23)),
        row=1, col=col,
    )

style_figure(fig, "", 1850, 880)
fig.update_annotations(font=dict(size=21, color=PLOT_TEXT_COLOR))

# Road shapefile: use ROADS_PATH if set, otherwise find it under DATA_DIR.
road_path = globals().get("ROADS_PATH")
if road_path is None:
    road_files = sorted(DATA_DIR.glob("**/*[Rr]oad*.shp"))
    road_path = road_files[0] if road_files else None
if road_path is None:
    raise FileNotFoundError("Set ROADS_PATH to the road shapefile.")

roads_local = (
    gpd.read_file(road_path)
    .to_crs(a2.rio.crs)
    .cx[map_bounds[0]:map_bounds[2], map_bounds[1]:map_bounds[3]]
)

road_x, road_y = [], []
for geometry in roads_local.geometry.dropna():
    parts = geometry.geoms if hasattr(geometry, "geoms") else [geometry]
    for part in parts:
        if hasattr(part, "xy"):
            xx, yy = part.xy
            road_x.extend([*xx, None])
            road_y.extend([*yy, None])

for col in (1, 2):
    fig.add_trace(go.Scatter(
        x=road_x, y=road_y,
        mode="lines",
        line=dict(
            color="rgba(40,55,70,.55)" if col == 1
                  else "rgba(240,245,250,.65)",
            width=.9,
        ),
        hoverinfo="skip",
        showlegend=False,
    ), row=1, col=col)

    fig.update_xaxes(
        tickmode="array",
        tickvals=[124.95, 125.00, 125.05],
        ticktext=["124.95°E", "125.00°E", "125.05°E"],
        tickfont=dict(size=15),
        showgrid=False,
        row=1, col=col,
    )
    fig.update_yaxes(
        tickmode="array",
        tickvals=[11.20, 11.25, 11.30],
        ticktext=["11.20°N", "11.25°N", "11.30°N"],
        tickfont=dict(size=15),
        showgrid=False,
        row=1, col=col,
    )

# (panel, label x, label y, anchor, text, text color)
label_positions = {
    "Pixel 23": (1, 124.925, 11.305, "left",  "Pixel 23", PLOT_TEXT_COLOR),
    "Pixel 21": (1, 124.925, 11.265, "left",  "Pixel 21", PLOT_TEXT_COLOR),
    "Pixel 13": (1, 124.925, 11.225, "left",  "Pixel 13", PLOT_TEXT_COLOR),
    "Pixel 22": (1, 125.083, 11.295, "right", "Pixel 22", PLOT_TEXT_COLOR),
    "Pixel 11": (1, 125.083, 11.250, "right", "Pixel 11", PLOT_TEXT_COLOR),
    "Pixel 12": (1, 125.083, 11.205, "right", "Pixel 12", PLOT_TEXT_COLOR),

    "Port": (2, 124.925, 11.290, "left", "Port", "white"),
    "Kassel City residential": (
        2, 124.925, 11.250, "left", "Kassel City residential", "white"
    ),
    "BLISS Sagkahan residential": (
        2, 124.925, 11.210, "left", "BLISS Sagkahan residential", "white"
    ),
    "Main-road vicinity": (
        2, 125.025, 11.270, "left", "Main-road vicinity", PLOT_TEXT_COLOR
    ),
    "Rizal Plaza": (
        2, 125.030, 11.249, "left", "Rizal Plaza", PLOT_TEXT_COLOR
    ),
    "Airport": (
        2, 125.083, 11.245, "right", "Airport", PLOT_TEXT_COLOR
    ),
    "Astrodome": (
        2, 125.083, 11.220, "right", "Astrodome", PLOT_TEXT_COLOR
    ),
    "Robinsons Place": (
        2, 125.083, 11.195, "right", "Robinsons Place", PLOT_TEXT_COLOR
    ),
}

for p in site_df.itertuples():
    # Outline the selected 1×1 pixel on BOTH maps.
    for col in (1, 2):
        for edge_color, edge_width in (("white", 4), ("black", 2)):
            fig.add_shape(
                type="rect",
                x0=p.x - dx / 2,
                x1=p.x + dx / 2,
                y0=p.y - dy / 2,
                y1=p.y + dy / 2,
                line=dict(color=edge_color, width=edge_width),
                fillcolor="rgba(0,0,0,0)",
                layer="above",
                row=1, col=col,
            )

    # Label each pixel on just one panel.
    col, label_x, label_y, anchor, label, color = label_positions[p.site]
    xref = "x" if col == 1 else "x2"
    yref = "y" if col == 1 else "y2"

    fig.add_annotation(
        x=p.x, y=p.y,
        xref=xref, yref=yref,
        ax=label_x, ay=label_y,
        axref=xref, ayref=yref,
        text=label,
        xanchor=anchor,
        showarrow=True,
        arrowhead=2,
        arrowsize=.7,
        arrowwidth=1,
        arrowcolor=color,
        bgcolor="rgba(0,0,0,0)",
        borderwidth=0,
        font=dict(
            size=16,
            color=color,
            shadow=(
                "0 0 3px #07152C, 0 0 6px #07152C"
                if color == "white"
                else "0 0 3px white, 0 0 6px white"
            ),
        ),
    )

for trace in fig.data:
    if isinstance(trace, go.Heatmap) and trace.showscale:
        trace.colorbar.tickfont = dict(size=16)
        trace.colorbar.title.font = dict(size=18)

fig.update_layout(
    margin=dict(l=80, r=135, t=75, b=55),
    showlegend=False,
)
finish_figure(fig, "poi_ghsl_ntl_pixel_boxes")

In [ ]:
# Extract daily and four-day RQ NTL for those selected pixels and windows.
# Run once; the following plot cells can then be rerun without re-extraction.

poi_daily_rows = []
poi_four_rows = []

first_block = int(
    (CHARACTERIZATION_START - EVENT_DATE).days // AGGREGATION_DAYS
)
last_block = int(
    (PRE_EVENT_END - EVENT_DATE).days // AGGREGATION_DAYS
)

for block in tqdm(
    range(first_block, last_block + 1),
    desc="Selected POI windows",
):
    start = EVENT_DATE + pd.Timedelta(
        days=block * AGGREGATION_DAYS
    )
    end = start + pd.Timedelta(days=AGGREGATION_DAYS - 1)
    midpoint = start + pd.Timedelta(
        days=(AGGREGATION_DAYS - 1) / 2
    )
    full_block = (
        start >= CHARACTERIZATION_START
        and end <= PRE_EVENT_END
    )
    indices = np.flatnonzero(day_blocks == block)

    if len(indices):
        fresh, capped, _ = read_days(indices)
        composite = median(capped, axis=0)
    else:
        composite = np.full(
            len(land_pixels), np.nan, dtype=np.float32
        )

    for site in site_df.site:
        for size in (1, 3, 5):
            support = poi_fixed[(site, size)]
            n_fixed = len(support)

            # Daily values, including days in an incomplete edge block.
            if len(indices):
                for j, date_index in enumerate(indices):
                    date = dates[date_index]
                    if not CHARACTERIZATION_START <= date <= PRE_EVENT_END:
                        continue

                    values = capped[j, support]
                    n_valid = int(np.isfinite(values).sum())
                    sc_pct = (
                        100 * n_valid / n_fixed
                        if n_fixed else np.nan
                    )
                    observed = (
                        n_valid > 0
                        and sc_pct >= SPATIAL_COMPLETENESS_PCT
                    )

                    poi_daily_rows.append(dict(
                        site=site,
                        window=size,
                        date=date,
                        raw_uncapped=median(fresh[j, support]),
                        ntl=median(values) if observed else np.nan,
                        sc_pct=sc_pct,
                        valid_pixels=n_valid,
                        n_fixed=n_fixed,
                        observed=observed,
                    ))

            # Four-day results use complete pre-Haiyan blocks only.
            if not full_block:
                continue

            values = composite[support]
            valid = np.isfinite(values)
            n_valid = int(valid.sum())
            sc_pct = (
                100 * n_valid / n_fixed
                if n_fixed else np.nan
            )
            observed = (
                n_valid > 0
                and sc_pct >= SPATIAL_COMPLETENESS_PCT
            )
            level = median(values) if observed else np.nan
            reference_sum = (
                ntl0[support][valid].sum()
                if n_valid else np.nan
            )

            poi_four_rows.append(dict(
                site=site,
                window=size,
                date=midpoint,
                date_start=start,
                date_end=end,
                ntl=level,
                rq_median=level,
                median_ntl=level,
                q25=(
                    np.quantile(values[valid], .25)
                    if observed else np.nan
                ),
                q75=(
                    np.quantile(values[valid], .75)
                    if observed else np.nan
                ),
                sc_pct=sc_pct,
                n_valid=n_valid,
                valid_pixels=n_valid,
                n_fixed=n_fixed,
                observed=observed,
                reference_pct=(
                    100 * values[valid].sum() / reference_sum
                    if observed and reference_sum > 0
                    else np.nan
                ),
            ))

poi_daily = pd.DataFrame(poi_daily_rows)
poi_four_day = pd.DataFrame(poi_four_rows)
poi_box = poi_four_day

print(
    f"{len(site_df)} selected pixels; "
    f"{len(poi_four_day)} four-day window records."
)

In [ ]:
# Four-day RQ NTL at selected 1×1 pixels

plot_size = 1
urban = site_df[site_df.ghsl.eq(30)].copy()
other = site_df[site_df.ghsl.ne(30)].copy()

missing_sites = set(site_df.site) - set(poi_box.site)
if missing_sites:
    raise ValueError(
        "Rerun the four-day extraction cell for: "
        + ", ".join(sorted(missing_sites))
    )

pixel_symbols = {
    23: "diamond", 22: "square", 21: "triangle-up",
    13: "circle", 12: "triangle-down", 11: "star",
}

def add_group_key(fig, group, columns, x0, width, heading):
    fig.add_annotation(
        x=x0, y=-.09, xref="paper", yref="paper",
        text=heading, showarrow=False, xanchor="left",
        font=dict(size=18, color=PLOT_TEXT_COLOR),
    )
    for i, p in enumerate(group.itertuples()):
        key_row, key_col = divmod(i, columns)
        x = x0 + key_col * width / columns
        y = -.16 - key_row * .065
        color = (
            p.color if p.ghsl == 30
            else GHSL_CLASS_COLORS[int(p.ghsl)]
        )
        fig.add_annotation(
            x=x, y=y, xref="paper", yref="paper",
            text="●", showarrow=False, xanchor="left",
            font=dict(size=24, color=color),
        )
        fig.add_annotation(
            x=x + .022, y=y, xref="paper", yref="paper",
            text=p.site, showarrow=False, xanchor="left",
            font=dict(size=16, color=PLOT_TEXT_COLOR),
        )

fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=.10,
    subplot_titles=[
        "Class 30 · named locations",
        "Other GHSL pixels · log1p display",
    ],
)

for col, group in [(1, urban), (2, other)]:
    for p in group.itertuples():
        g = poi_box[
            poi_box.site.eq(p.site)
            & poi_box.window.eq(plot_size)
        ].sort_values("date")

        color = (
            p.color if col == 1
            else GHSL_CLASS_COLORS[int(p.ghsl)]
        )
        symbol = (
            p.symbol if col == 1
            else pixel_symbols.get(int(p.ghsl), "circle")
        )
        display_y = (
            g.median_ntl if col == 1
            else np.log1p(g.median_ntl)
        )

        fig.add_trace(
            go.Scatter(
                x=g.date,
                y=display_y,
                mode="lines+markers",
                line=dict(color=color, width=2.1),
                marker=dict(
                    color=color, symbol=symbol, size=4,
                    line=dict(color=PLOT_TEXT_COLOR, width=.35),
                ),
                opacity=.85,
                connectgaps=False,
                name=p.site,
                showlegend=False,
                customdata=np.column_stack([
                    g.median_ntl, g.sc_pct, g.n_valid
                ]),
                hovertemplate=(
                    f"{p.site}<br>%{{x|%d %b %Y}}"
                    "<br>Four-day RQ NTL %{customdata[0]:.2f}"
                    "<br>SC %{customdata[1]:.0f}%"
                    "<br>Valid pixels %{customdata[2]:.0f}"
                    "<extra></extra>"
                ),
            ),
            row=1, col=col,
        )

    fig.update_xaxes(
        range=[CHARACTERIZATION_START, PRE_EVENT_END],
        tickformat="%b %Y", dtick="M3",
        tickfont=dict(size=15),
        row=1, col=col,
    )

urban_values = poi_box[
    poi_box.site.isin(urban.site)
    & poi_box.window.eq(plot_size)
].median_ntl.dropna()
urban_max = (
    max(5, np.ceil(urban_values.max() * 1.05))
    if len(urban_values) else 5
)
fig.update_yaxes(
    range=[0, urban_max],
    nticks=6,
    title_text="NTL (nW cm⁻² sr⁻¹)",
    tickfont=dict(size=15),
    row=1, col=1,
)

other_values = poi_box[
    poi_box.site.isin(other.site)
    & poi_box.window.eq(plot_size)
].median_ntl.dropna()
other_max = (
    max(1.2, np.ceil(other_values.max() * 1.05))
    if len(other_values) else 1.2
)
raw_ticks = [
    value for value in (0, .25, .5, 1, 2, 5, 10, 15, 20)
    if value <= other_max
]
fig.update_yaxes(
    range=[0, np.log1p(other_max)],
    tickmode="array",
    tickvals=np.log1p(raw_ticks),
    ticktext=[str(value) for value in raw_ticks],
    title_text="NTL (nW cm⁻² sr⁻¹) · log1p axis",
    tickfont=dict(size=15),
    row=1, col=2,
)

style_figure(fig, "", 1840, 920)
fig.update_layout(
    margin=dict(l=90, r=55, t=80, b=290),
    showlegend=False,
    hovermode="closest",
)
fig.update_annotations(font=dict(size=19, color=PLOT_TEXT_COLOR))
add_group_key(fig, urban, 2, 0, .48, "Named class-30 locations")
add_group_key(fig, other, 3, .55, .45, "Individual GHSL pixels")
finish_figure(fig, "poi_four_day_grouped_lines_v3")

## Interpretation ledger

| Evidence pattern | What it would support | What it would **not** establish |
|---|---|---|
| Low regional NTL coincides with low SC and dimmer reference values among visible pixels | Changing pixel composition / observability contributes to the trough | That every qualified pixel dimmed |
| Low day has good SC, normal reference brightness among observed pixels, and many source values at 0.01 | A retrieval floor or common radiance problem warrants product-level inspection | A regional power interruption |
| Similar peak in bright NTL, dim NTL, SC or the observation indicator | A shared observing/processing mechanism merits testing | A specifically lunar cause |
| Peak persists in high-support bright pixels but not dim pixels or SC | A location- or brightness-dependent process is plausible | Seasonal electricity demand without an external comparator |
| Candidate breakpoint is near a documented local event and persists without interpolation | A nonstationary baseline requiring event annotation or sensitivity analysis | A validated disaster mechanism from NTL alone |

**Decision before Notebook 07:** retain a dynamic baseline only if its main period and trend are stable to observability restrictions, fixed-pixel comparison, interpolation choices and sensible STL periods. Otherwise show a simpler baseline alongside the dynamic candidate and propagate baseline uncertainty into impact/recovery metrics. This notebook provides diagnostics, not an automatic pass/fail decision.

**Method sources:** [STL forecasting distinction](https://pkg.robjhyndman.com/forecast/reference/forecast.stl.html); [BFAST R documentation](https://search.r-project.org/CRAN/refmans/bfast/html/bfast.html). The first describes forecasting seasonally adjusted values then re-seasonalizing; the latter is a break-detection method. Neither makes an assumed 28-day NTL period physically real.